In [31]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [34]:
SEED = 1990
df_prototype = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/prototype.csv")
df_test = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/test.csv")
pkl_file_path = '/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/features/I3D_features.pkl'
df_features = pd.read_pickle(pkl_file_path)
training_matrix = {}


le = LabelEncoder()
df_prototype["encoded_category"] = le.fit_transform(df_prototype["category"].astype(str))

rows = []
for video_id, enc_cat in zip(df_prototype["uid"].values, df_prototype["encoded_category"].values):
    tensor = (
        df_features.loc[df_features["id"] == video_id, "I3D_features"]
        .iloc[0]
        .squeeze()
    )
    rows.append((video_id, tensor, enc_cat))

df_train = pd.DataFrame(rows, columns=["uid", "I3D_features", "encoded_category"])
df_train.head()
df_train.shape

(4765, 3)

In [35]:
df_prototype.shape

(4765, 5)

In [36]:
counts = df_train["encoded_category"].value_counts()
ok_cats = counts[counts >= 2].index

df_ok = df_train[df_train["encoded_category"].isin(ok_cats)].copy()
df_singletons = df_train[~df_train["encoded_category"].isin(ok_cats)].copy()

df_tr, df_val = train_test_split(
    df_ok,
    test_size=0.2,
    random_state=SEED,
    stratify=df_ok["encoded_category"]
)


In [37]:
print(df_tr.head())
print(df_tr.shape)
print(df_val.head())
print(df_val.shape)

                uid                                       I3D_features  \
1342    CPPp4bkc2M0  [[0.0053843306, 0.0023832985, 0.0006706109, 0....   
2024  KIoc3qEV7No_1  [[0.024400018, 0.07057263, 0.12738055, 0.12688...   
2977    WQpetmGGa2Y  [[0.0151490215, 0.0, 0.0, 0.0, 0.0, 0.0, 0.002...   
1618    F7s5GtLbtGE  [[0.04165503, 0.043686528, 0.05677606, 0.06055...   
4280  rZnNOW47uNk_1  [[0.017665822, 0.040160134, 0.04929811, 0.0423...   

      encoded_category  
1342                 1  
2024                42  
2977                 1  
1618                19  
4280                 3  
(3811, 3)
                uid                                       I3D_features  \
1182    AZ71WPBDNuM  [[0.09885164, 0.10414533, 0.09108694, 0.032520...   
3655    g8ZrP9KZGlk  [[0.0646083, 0.007057772, 0.004027713, 0.00205...   
3995  mKT_yIIVqbA_1  [[0.050534815, 0.054080438, 0.04703696, 0.0473...   
1385    CpbiUt_BR3U  [[0.028324492, 0.003112904, 0.0051280404, 0.00...   
1151    ADgKP4UpMkg  [[0.

In [41]:
x = df_tr.loc[1342, "I3D_features"]
print(len(x))
type(x[0]), len(x[0])


1024


(numpy.ndarray, 31)

In [63]:
print(len(df_tr["I3D_features"].apply(lambda x: len(x[0])).unique()))


45


In [54]:
def pad_features(features, T_max=51):
    D = 1024
    T = features.shape[1]  # safer than len(features[0])

    if T > T_max:
        features = features[:, :T_max]
        T = T_max

    padded = np.zeros((D, T_max), dtype=np.float32)
    padded[:, :T] = features[:, :T]

    mask = np.zeros(T_max, dtype=np.float32)
    mask[:T] = 1.0

    return padded, mask, T


In [51]:
class DFVideoDataset(Dataset):
    def __init__(self, df, label_col="encoded_category"):
        self.df = df.reset_index(drop=True)
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        x = self.df.loc[idx, "I3D_features"]  # often list-of-arrays
        x = np.array(x)                       # -> (1024, T) if uniform
        assert x.ndim == 2 and x.shape[0] == 1024, f"bad shape at idx={idx}: {x.shape}"

        y = self.df.loc[idx, self.label_col]
        return x, y


In [55]:
def collate_fn(batch, T_max=51):
    batch_size = len(batch)
    features_list = [item[0] for item in batch]
    labels = [item[1] for item in batch]

    padded_batch = np.zeros((batch_size, 1024, T_max), dtype=np.float32)
    mask_batch   = np.zeros((batch_size, T_max), dtype=np.float32)

    for i, feat in enumerate(features_list):
        padded, mask, T = pad_features(feat, T_max=T_max)
        padded_batch[i] = padded
        mask_batch[i] = mask

    return (
        torch.from_numpy(padded_batch),
        torch.from_numpy(mask_batch),
        torch.tensor(labels, dtype=torch.long),
    )


In [ ]:
'''ds_tr = DFVideoDataset(df_tr, label_col="encoded_category")   # <-- update label col name
ds_val = DFVideoDataset(df_val, label_col="encoded_category") 
# 1) pick 1 sample per class WITHOUT resetting index
df_val_support = (
    df_val
    .groupby("encoded_category", group_keys=False)
    .apply(lambda g: g.sample(n=1, random_state=SEED))
)

# 2) drop using the original indices
df_val_query = df_val.drop(index=df_val_support.index)

# 3) now reset indexes (optional, for cleanliness)
df_val_support = df_val_support.reset_index(drop=True)
df_val_query   = df_val_query.reset_index(drop=True)


print("val total:", len(df_val))
print("support:", len(df_val_support), " (should be = #classes in val)")
print("query:", len(df_val_query))
print("classes in support:", df_val_support["encoded_category"].nunique())


# Build proper datasets
ds_tr = DFVideoDataset(df_tr, label_col="encoded_category")

ds_val_support = DFVideoDataset(df_val_support, label_col="encoded_category")
ds_val_query   = DFVideoDataset(df_val_query,   label_col="encoded_category")

# Build loaders
train_loader = DataLoader(ds_tr, batch_size=8, shuffle=True,  collate_fn=collate_fn)
val_support_loader = DataLoader(ds_val_support, batch_size=8, shuffle=False, collate_fn=collate_fn)
val_query_loader   = DataLoader(ds_val_query,   batch_size=8, shuffle=False, collate_fn=collate_fn)


x, m, y = next(iter(loader))
print("x:", x.shape, x.dtype)
print("m:", m.shape, m.dtype)
print("y:", y.shape, y.dtype)
print("mask sums:", m.sum(dim=1))
'''

val total: 953
support: 57  (should be = #classes in val)
query: 896
classes in support: 57
x: torch.Size([8, 1024, 51]) torch.float32
m: torch.Size([8, 51]) torch.float32
y: torch.Size([8]) torch.int64
mask sums: tensor([ 9.,  6., 12.,  7.,  6., 15., 14., 12.])


/var/folders/4r/h8jqnrvd187321k0xl5n0g4h0000gn/T/ipykernel_20469/617202107.py:7: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=1, random_state=SEED))


In [77]:
k = 5
cols = ["uid", "I3D_features", "encoded_category"]

vc = df_val["encoded_category"].value_counts()
eligible_classes = vc[vc >= (k + 1)].index
df_val_k = df_val[df_val["encoded_category"].isin(eligible_classes)].copy()

print("Eligible classes:", df_val_k["encoded_category"].nunique(), "/", df_val["encoded_category"].nunique())
print("Dropped samples:", len(df_val) - len(df_val_k))

# k-shot support (no FutureWarning)
df_val_support = (
    df_val_k[cols]
    .groupby("encoded_category", group_keys=False)
    .sample(n=k, random_state=SEED)
)

# query = remaining
df_val_query = df_val_k.drop(index=df_val_support.index)

# reset indices (optional)
df_val_support = df_val_support.reset_index(drop=True)
df_val_query   = df_val_query.reset_index(drop=True)

print("support:", len(df_val_support), "should be", k * df_val_support["encoded_category"].nunique())
print("query:", len(df_val_query))
print("classes support:", df_val_support["encoded_category"].nunique())
print("classes query:", df_val_query["encoded_category"].nunique())

# build datasets (important)
ds_val_support = DFVideoDataset(df_val_support, label_col="encoded_category")
ds_val_query   = DFVideoDataset(df_val_query,   label_col="encoded_category")

# loaders
val_support_loader = DataLoader(ds_val_support, batch_size=8, shuffle=False, collate_fn=collate_fn)
val_query_loader   = DataLoader(ds_val_query,   batch_size=8, shuffle=False, collate_fn=collate_fn)


Eligible classes: 45 / 57
Dropped samples: 43
support: 225 should be 225
query: 685
classes support: 45
classes query: 45


In [106]:
proto_cats = set(df_prototype["category"].astype(str))
test_cats  = set(df_test["category"].astype(str))

missing_in_proto = sorted(test_cats - proto_cats)
print("Test categories missing in prototype:", missing_in_proto)
print("#missing:", len(missing_in_proto))
df_test_seen = df_test[df_test["category"].astype(str).isin(proto_cats)].copy()

print("Original test:", df_test.shape)
print("Filtered  test:", df_test_seen.shape)
print("Filtered-out rows:", df_test.shape[0] - df_test_seen.shape[0])


Test categories missing in prototype: ['goal']
#missing: 1
Original test: (2285, 4)
Filtered  test: (2284, 4)
Filtered-out rows: 1


In [115]:
# ---- Build full prototype dataframe with features ----
proto_rows = []
for uid, enc_cat in zip(df_prototype["uid"].values, df_prototype["encoded_category"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    proto_rows.append((uid, tensor, enc_cat))
df_proto_full = pd.DataFrame(proto_rows, columns=["uid", "I3D_features", "encoded_category"])

# ---- Build full test dataframe with features ----
# IMPORTANT: use the SAME label encoder fitted on prototype categories
df_test = df_test.copy()

# IMPORTANT: encode ONLY the filtered test set
df_test_seen = df_test[df_test["category"].astype(str).isin(proto_cats)].copy()
df_test_seen["encoded_category"] = le.transform(df_test_seen["category"].astype(str))

# Build full test dataframe with features
test_rows = []
for uid, enc_cat in zip(df_test_seen["uid"].values, df_test_seen["encoded_category"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    test_rows.append((uid, tensor, enc_cat))

df_test_full = pd.DataFrame(test_rows, columns=["uid", "I3D_features", "encoded_category"])



print("proto shape:", df_proto_full.shape, "classes:", df_proto_full["encoded_category"].nunique())
print("test shape:", df_test_full.shape, "classes:", df_test_full["encoded_category"].nunique())
test_dataset = DFVideoDataset(df_test_full, label_col="encoded_category")
proto_dataset = DFVideoDataset(df_proto_full, label_col="encoded_category")

proto_loader = DataLoader(proto_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, collate_fn=collate_fn)



proto shape: (4765, 3) classes: 58
test shape: (2284, 3) classes: 57


In [80]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------
# Stage-1 Encoder (d=256)
# -------------------------
class MaskedMeanEncoder(nn.Module):
    """
    Input:
        x: (B, 1024, T)
        mask: (B, T)  float {0,1}
    Output:
        z: (B, 256) L2-normalized
    """
    def __init__(self, d=256, use_layernorm=True):
        super().__init__()
        self.d = d
        self.use_layernorm = use_layernorm

        # Project each timestep: 1024 -> d
        self.proj = nn.Linear(1024, d, bias=True)

        # Optional normalization for stability
        self.ln = nn.LayerNorm(1024) if use_layernorm else None

    def forward(self, x, mask):
        # x: (B, 1024, T) -> (B, T, 1024)
        x = x.transpose(1, 2)  # (B, T, 1024)

        if self.ln is not None:
            x = self.ln(x)  # layernorm over last dim (1024)

        # Per-timestep projection: (B, T, 1024) -> (B, T, d)
        h = self.proj(x)  # (B, T, d)

        # Masked mean pooling over time
        # mask: (B, T) -> (B, T, 1)
        mask_ = mask.unsqueeze(-1)  # (B, T, 1)
        h = h * mask_               # zero out padded timesteps

        denom = mask_.sum(dim=1).clamp_min(1.0)  # (B, 1)
        z = h.sum(dim=1) / denom                 # (B, d)

        # L2 normalize for cosine similarity stability
        z = F.normalize(z, p=2, dim=-1)
        return z


class EncoderWithHead(nn.Module):
    """
    Train-time wrapper: encoder + classification head.
    """
    def __init__(self, num_classes, d=256):
        super().__init__()
        self.encoder = MaskedMeanEncoder(d=d, use_layernorm=True)
        self.head = nn.Linear(d, num_classes)

    def forward(self, x, mask):
        z = self.encoder(x, mask)       # (B, d)
        logits = self.head(z)           # (B, C)
        return logits, z


# -------------------------
# Utility: build prototypes (from a loader)
# -------------------------
@torch.no_grad()
def compute_class_prototypes(encoder, loader, device):
    """
    Builds mean embedding per class from samples in loader.
    Returns:
        proto_mat: (C, d) L2-normalized
    """
    encoder.eval()

    sums = {}
    counts = {}

    for x, m, y in loader:
        x = x.to(device)
        m = m.to(device)
        y = y.to(device)

        z = encoder(x, m)  # (B, d)

        for i in range(z.size(0)):
            cls = int(y[i].item())
            if cls not in sums:
                sums[cls] = z[i].detach().clone()
                counts[cls] = 1
            else:
                sums[cls] += z[i]
                counts[cls] += 1

    # Build ordered prototype matrix
    classes = sorted(sums.keys())
    proto = []
    for c in classes:
        p = sums[c] / max(counts[c], 1)
        p = F.normalize(p, p=2, dim=-1)
        proto.append(p)

    proto_mat = torch.stack(proto, dim=0)  # (C, d)
    return classes, proto_mat


# -------------------------
# Utility: retrieval evaluation (Top-1/5/10)
# -------------------------
@torch.no_grad()
def retrieval_eval(encoder, query_loader, proto_classes, proto_mat, device, topk=(1,5,10)):
    """
    query embeddings vs prototype matrix by cosine similarity.
    """
    encoder.eval()
    proto_mat = proto_mat.to(device)  # (C, d)

    topk_hits = {k: 0 for k in topk}
    total = 0

    for x, m, y in query_loader:
        x = x.to(device)
        m = m.to(device)
        y = y.to(device)

        z = encoder(x, m)  # (B, d)

        # cosine sim because vectors are L2-normalized: sim = z @ proto.T
        sims = z @ proto_mat.t()  # (B, C)
        total += z.size(0)

        # Map true labels to prototype indices
        # proto_classes is sorted list of class ids used in proto_mat
        # We'll make a dict for quick lookup (small C=57 so fine)
        idx_map = {c:i for i,c in enumerate(proto_classes)}
        true_idx = torch.tensor([idx_map[int(t.item())] for t in y], device=device)

        # compute top-k
        for k in topk:
            pred_idx = sims.topk(k, dim=1).indices  # (B, k)
            hit = (pred_idx == true_idx.unsqueeze(1)).any(dim=1).sum().item()
            topk_hits[k] += hit

    accs = {k: topk_hits[k]/max(total,1) for k in topk}
    return accs


# -------------------------
# Training loop (classification loss)
# -------------------------
from tqdm.auto import tqdm

def train_stage1(
    train_loader,
    val_support_loader,
    val_query_loader,
    num_classes,
    epochs=10,
    lr=1e-3,
    device="cuda" if torch.cuda.is_available() else "mps"
):
    model = EncoderWithHead(num_classes=num_classes, d=256).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)

        for x, m, y in pbar:
            x = x.to(device)
            m = m.to(device)
            y = y.to(device)

            logits, _ = model(x, m)
            loss = criterion(logits, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_seen += bs
            total_correct += (logits.argmax(dim=1) == y).sum().item()

            # live stats
            pbar.set_postfix(
                loss=total_loss / max(total_seen, 1),
                acc=total_correct / max(total_seen, 1)
            )

        train_loss = total_loss / max(total_seen, 1)
        train_acc  = total_correct / max(total_seen, 1)

        # Retrieval-style validation
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs = retrieval_eval(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)

        print(
            f"Epoch {ep:02d} | "
            f"TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | "
            f"ValRetr Top1={val_accs.get(1,0):.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f}"
        )

    return model



In [81]:
num_classes = df_train["encoded_category"].nunique()

model = train_stage1(
    train_loader,
    val_support_loader,
    val_query_loader,
    num_classes=num_classes,
    epochs=10,
    lr=1e-3,
    device="cuda" if torch.cuda.is_available() else "mps"
)


Epoch 01 | TrainLoss=3.6792 TrainAcc=0.1425 | ValRetr Top1=0.1285 Top5=0.2715 Top10=0.4015


Epoch 02 | TrainLoss=3.4066 TrainAcc=0.1892 | ValRetr Top1=0.1343 Top5=0.2905 Top10=0.4263


Epoch 03 | TrainLoss=3.2534 TrainAcc=0.2183 | ValRetr Top1=0.1343 Top5=0.2993 Top10=0.4350


Epoch 04 | TrainLoss=3.1327 TrainAcc=0.2346 | ValRetr Top1=0.1255 Top5=0.3036 Top10=0.4453


Epoch 05 | TrainLoss=3.0355 TrainAcc=0.2558 | ValRetr Top1=0.1241 Top5=0.3182 Top10=0.4453


Epoch 06 | TrainLoss=2.9428 TrainAcc=0.2695 | ValRetr Top1=0.1299 Top5=0.3109 Top10=0.4613


Epoch 07 | TrainLoss=2.8604 TrainAcc=0.2834 | ValRetr Top1=0.1226 Top5=0.3168 Top10=0.4803


Epoch 08 | TrainLoss=2.7819 TrainAcc=0.3036 | ValRetr Top1=0.1109 Top5=0.2832 Top10=0.4584


Epoch 09 | TrainLoss=2.6989 TrainAcc=0.3157 | ValRetr Top1=0.1299 Top5=0.3328 Top10=0.4876


Epoch 10 | TrainLoss=2.6383 TrainAcc=0.3311 | ValRetr Top1=0.1124 Top5=0.3139 Top10=0.4759


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedAttnEncoder(nn.Module):
    """
    Input:
        x: (B, 1024, T)
        mask: (B, T) float {0,1}
    Output:
        z: (B, d) L2-normalized
    """
    def __init__(self, d=256, use_layernorm=True):
        super().__init__()
        self.d = d
        self.ln = nn.LayerNorm(1024) if use_layernorm else None

        # Per-timestep projection: 1024 -> d
        self.proj = nn.Linear(1024, d)

        # Attention scoring: d -> 1 (per timestep)
        # (Simple & effective. We can make it 2-layer later if needed.)
        self.attn = nn.Linear(d, 1)
        self.drop = nn.Dropout(p=0.05)


    def forward(self, x, mask):
        # x: (B, 1024, T) -> (B, T, 1024)
        x = x.transpose(1, 2)  # (B, T, 1024)

        if self.ln is not None:
            x = self.ln(x)

        # h: (B, T, d)
        h = self.proj(x)
        h = self.drop(h)

        # scores: (B, T)
        scores = self.attn(torch.tanh(h)).squeeze(-1)

        # mask padded positions: set to very negative so softmax -> 0 there
        # mask is float {0,1}; valid positions are 1
        scores = scores.masked_fill(mask == 0, -1e9)

        # alpha: (B, T)
        alpha = F.softmax(scores, dim=1)

        # weighted sum: (B, d)
        z = torch.sum(h * alpha.unsqueeze(-1), dim=1)

        # L2 normalize
        z = F.normalize(z, p=2, dim=-1)
        return z
class EncoderWithHead(nn.Module):
    def __init__(self, num_classes, d=256):
        super().__init__()
        self.encoder = MaskedAttnEncoder(d=d, use_layernorm=True)
        self.head = nn.Linear(d, num_classes)

    def forward(self, x, mask):
        z = self.encoder(x, mask)
        logits = self.head(z)
        return logits, z


# -------------------------
# Utility: build prototypes (from a loader)
# -------------------------
@torch.no_grad()
def compute_class_prototypes(encoder, loader, device):
    """
    Builds mean embedding per class from samples in loader.
    Returns:
        proto_mat: (C, d) L2-normalized
    """
    encoder.eval()

    sums = {}
    counts = {}

    for x, m, y in loader:
        x = x.to(device)
        m = m.to(device)
        y = y.to(device)

        z = encoder(x, m)  # (B, d)

        for i in range(z.size(0)):
            cls = int(y[i].item())
            if cls not in sums:
                sums[cls] = z[i].detach().clone()
                counts[cls] = 1
            else:
                sums[cls] += z[i]
                counts[cls] += 1

    # Build ordered prototype matrix
    classes = sorted(sums.keys())
    proto = []
    for c in classes:
        p = sums[c] / max(counts[c], 1)
        p = F.normalize(p, p=2, dim=-1)
        proto.append(p)

    proto_mat = torch.stack(proto, dim=0)  # (C, d)
    return classes, proto_mat


# -------------------------
# Utility: retrieval evaluation (Top-1/5/10)
# -------------------------
@torch.no_grad()
def retrieval_eval(encoder, query_loader, proto_classes, proto_mat, device, topk=(1,5,10)):
    """
    query embeddings vs prototype matrix by cosine similarity.
    """
    encoder.eval()
    proto_mat = proto_mat.to(device)  # (C, d)

    topk_hits = {k: 0 for k in topk}
    total = 0

    for x, m, y in query_loader:
        x = x.to(device)
        m = m.to(device)
        y = y.to(device)ß

        z = encoder(x, m)  # (B, d)

        # cosine sim because vectors are L2-normalized: sim = z @ proto.T
        sims = z @ proto_mat.t()  # (B, C)
        total += z.size(0)

        # Map true labels to prototype indices
        # proto_classes is sorted list of class ids used in proto_mat
        # We'll make a dict for quick lookup (small C=57 so fine)
        idx_map = {c:i for i,c in enumerate(proto_classes)}
        true_idx = torch.tensor([idx_map[int(t.item())] for t in y], device=device)

        # compute top-k
        for k in topk:
            pred_idx = sims.topk(k, dim=1).indices  # (B, k)
            hit = (pred_idx == true_idx.unsqueeze(1)).any(dim=1).sum().item()
            topk_hits[k] += hit

    accs = {k: topk_hits[k]/max(total,1) for k in topk}
    return accs


# -------------------------
# Training loop (classification loss)
# -------------------------
from tqdm.auto import tqdm
import copy
from tqdm.auto import tqdm

def train_stage1(
    train_loader,
    val_support_loader,
    val_query_loader,
    num_classes,
    epochs=30,
    lr=1e-4,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
):
    model = EncoderWithHead(num_classes=num_classes, d=256).to(device)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=0.05   # start here
    )
    criterion = nn.CrossEntropyLoss()

    best_top1 = -1.0
    best_epoch = -1
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)

        for x, m, y in pbar:
            x = x.to(device)
            m = m.to(device)
            y = y.to(device)

            logits, _ = model(x, m)
            loss = criterion(logits, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_seen += bs
            total_correct += (logits.argmax(dim=1) == y).sum().item()

            pbar.set_postfix(
                loss=total_loss / max(total_seen, 1),
                acc=total_correct / max(total_seen, 1)
            )

        train_loss = total_loss / max(total_seen, 1)
        train_acc  = total_correct / max(total_seen, 1)

        # retrieval validation
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs = retrieval_eval(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)
        top1 = float(val_accs.get(1, 0.0))

        print(
            f"Epoch {ep:02d} | "
            f"TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | "
            f"ValRetr Top1={val_accs.get(1,0):.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f}"
        )

        # ---- Early stopping logic ----
        if top1 > best_top1 + min_delta:
            best_top1 = top1
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"Early stopping at epoch {ep} (best Top1={best_top1:.4f} at epoch {best_epoch}).")
            break

    # restore best weights
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} (Top1={best_top1:.4f}).")

    return model



In [102]:
num_classes = df_train["encoded_category"].nunique()

num_classes = df_train["encoded_category"].nunique()

model = train_stage1(
    train_loader,
    val_support_loader,
    val_query_loader,
    num_classes=num_classes,
    epochs=30,
    lr=1e-4,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
)



Epoch 01 | TrainLoss=3.9303 TrainAcc=0.1110 | ValRetr Top1=0.1212 Top5=0.2482 Top10=0.3664


Epoch 02 | TrainLoss=3.7993 TrainAcc=0.1427 | ValRetr Top1=0.1474 Top5=0.2438 Top10=0.3956


Epoch 03 | TrainLoss=3.7114 TrainAcc=0.1567 | ValRetr Top1=0.1533 Top5=0.2642 Top10=0.3985


Epoch 04 | TrainLoss=3.6418 TrainAcc=0.1656 | ValRetr Top1=0.1547 Top5=0.2774 Top10=0.4102


Epoch 05 | TrainLoss=3.5815 TrainAcc=0.1711 | ValRetr Top1=0.1606 Top5=0.2861 Top10=0.4438


...

Epoch 07 | TrainLoss=3.4782 TrainAcc=0.1845 | ValRetr Top1=0.1562 Top5=0.2993 Top10=0.4482


Epoch 08 | TrainLoss=3.4330 TrainAcc=0.1973 | ValRetr Top1=0.1372 Top5=0.3036 Top10=0.4482


Epoch 09 | TrainLoss=3.3899 TrainAcc=0.2110 | ValRetr Top1=0.1518 Top5=0.3197 Top10=0.4628


Epoch 10 | TrainLoss=3.3487 TrainAcc=0.2152 | ValRetr Top1=0.1460 Top5=0.3036 Top10=0.4628


Epoch 11 | TrainLoss=3.3124 TrainAcc=0.2254 | ValRetr Top1=0.1489 Top5=0.3255 Top10=0.4657
Early stopping at epoch 11 (best Top1=0.1693 at epoch 6).
Restored best model from epoch 6 (Top1=0.1693).


In [109]:
num_classes = df_train["encoded_category"].nunique()

model = train_stage1(
    train_loader,
    val_support_loader,
    val_query_loader,
    num_classes=num_classes,
    epochs=30,
    lr=1e-4,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
)


Epoch 01 | TrainLoss=3.9188 TrainAcc=0.1128 | ValRetr Top1=0.1095 Top5=0.2394 Top10=0.3620


Epoch 02 | TrainLoss=3.7900 TrainAcc=0.1441 | ValRetr Top1=0.1285 Top5=0.2511 Top10=0.3956


Epoch 03 | TrainLoss=3.7043 TrainAcc=0.1590 | ValRetr Top1=0.1460 Top5=0.2686 Top10=0.4058


Epoch 04 | TrainLoss=3.6349 TrainAcc=0.1674 | ValRetr Top1=0.1489 Top5=0.2715 Top10=0.4161


Epoch 05 | TrainLoss=3.5749 TrainAcc=0.1763 | ValRetr Top1=0.1577 Top5=0.2876 Top10=0.4248


...

Epoch 07 | TrainLoss=3.4718 TrainAcc=0.1931 | ValRetr Top1=0.1562 Top5=0.2949 Top10=0.4423


Epoch 08 | TrainLoss=3.4270 TrainAcc=0.2023 | ValRetr Top1=0.1518 Top5=0.3168 Top10=0.4482


Epoch 09 | TrainLoss=3.3856 TrainAcc=0.2089 | ValRetr Top1=0.1620 Top5=0.3168 Top10=0.4540


Epoch 10 | TrainLoss=3.3451 TrainAcc=0.2149 | ValRetr Top1=0.1460 Top5=0.3182 Top10=0.4584


Epoch 11 | TrainLoss=3.3076 TrainAcc=0.2204 | ValRetr Top1=0.1547 Top5=0.3139 Top10=0.4672
Early stopping at epoch 11 (best Top1=0.1635 at epoch 6).
Restored best model from epoch 6 (Top1=0.1635).


In [116]:
device = "cuda" if torch.cuda.is_available() else "mps"

# Build prototypes from FULL prototype set
proto_classes, proto_mat = compute_class_prototypes(model.encoder, proto_loader, device=device)

# Evaluate retrieval on FULL test set
test_accs = retrieval_eval(model.encoder, test_loader, proto_classes, proto_mat, device=device)

print("TEST Retrieval Accuracies:")
print("Top-1 :", round(test_accs.get(1, 0.0), 4))
print("Top-5 :", round(test_accs.get(5, 0.0), 4))
print("Top-10:", round(test_accs.get(10, 0.0), 4))


TEST Retrieval Accuracies:
Top-1 : 0.0841
Top-5 : 0.2671
Top-10: 0.4238


In [123]:
from tqdm.auto import tqdm
import copy
import torch
import torch.nn as nn

@torch.no_grad()
def eval_ce(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total_seen = 0.0, 0, 0
    for x, m, y in loader:
        x, m, y = x.to(device), m.to(device), y.to(device)
        logits, _ = model(x, m)
        loss = criterion(logits, y)
        bs = x.size(0)
        total_loss += loss.item() * bs
        total_correct += (logits.argmax(1) == y).sum().item()
        total_seen += bs
    return total_loss / max(total_seen, 1), total_correct / max(total_seen, 1)

def train_stage_gloss_ce(
    train_loader,
    val_loader,
    num_classes,
    epochs=30,
    lr=1e-4,
    weight_decay=0.01,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
):
    model = EncoderWithHead(num_classes=num_classes, d=256).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_val_loss = float("inf")
    best_epoch = -1
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_correct, total_seen = 0.0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)

        for x, m, y in pbar:
            x, m, y = x.to(device), m.to(device), y.to(device)
            logits, _ = model(x, m)
            loss = criterion(logits, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_correct += (logits.argmax(1) == y).sum().item()
            total_seen += bs
            pbar.set_postfix(loss=total_loss/max(total_seen,1), acc=total_correct/max(total_seen,1))

        train_loss = total_loss / max(total_seen, 1)
        train_acc = total_correct / max(total_seen, 1)

        val_loss, val_acc = eval_ce(model, val_loader, criterion, device)

        print(f"Epoch {ep:02d} | TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | ValLoss={val_loss:.4f} ValAcc={val_acc:.4f}")

        # early stopping on val loss (more stable than val acc here)
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"Early stopping at epoch {ep} (best ValLoss={best_val_loss:.4f} at epoch {best_epoch}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} (ValLoss={best_val_loss:.4f}).")

    return model


In [117]:
# Normalize gloss to avoid case/spacing mismatch
def normalize_gloss(g):
    return " ".join(str(g).strip().upper().split())

df_prototype = df_prototype.copy()
df_test = df_test.copy()
df_prototype["gloss_norm"] = df_prototype["gloss"].apply(normalize_gloss)
df_test["gloss_norm"] = df_test["gloss"].apply(normalize_gloss)

gloss_le = LabelEncoder()
df_prototype["encoded_gloss"] = gloss_le.fit_transform(df_prototype["gloss_norm"])
proto_glosses = set(df_prototype["gloss_norm"])
df_test_seen = df_test[df_test["gloss_norm"].isin(proto_glosses)].copy()
df_test_seen["encoded_gloss"] = gloss_le.transform(df_test_seen["gloss_norm"])

print("Original test:", df_test.shape)
print("Filtered test:", df_test_seen.shape)
print("Dropped:", df_test.shape[0] - df_test_seen.shape[0])


Original test: (2285, 5)
Filtered test: (2285, 6)
Dropped: 0


In [118]:
# prototype full
proto_rows = []
for uid, enc_g in zip(df_prototype["uid"].values, df_prototype["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    proto_rows.append((uid, tensor, enc_g))
df_proto_full = pd.DataFrame(proto_rows, columns=["uid", "I3D_features", "encoded_gloss"])

# test full
test_rows = []
for uid, enc_g in zip(df_test_seen["uid"].values, df_test_seen["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    test_rows.append((uid, tensor, enc_g))
df_test_full = pd.DataFrame(test_rows, columns=["uid", "I3D_features", "encoded_gloss"])


In [119]:
proto_dataset = DFVideoDataset(df_proto_full.rename(columns={"encoded_gloss":"encoded_category"}), label_col="encoded_category")
test_dataset  = DFVideoDataset(df_test_full.rename(columns={"encoded_gloss":"encoded_category"}),  label_col="encoded_category")

proto_loader = DataLoader(proto_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, collate_fn=collate_fn)


In [120]:
device = "cuda" if torch.cuda.is_available() else "mps"

proto_classes, proto_mat = compute_class_prototypes(model.encoder, proto_loader, device=device)
test_accs = retrieval_eval(model.encoder, test_loader, proto_classes, proto_mat, device=device)

print("TEST (Gloss Retrieval) Accuracies:")
print("Top-1 :", round(test_accs.get(1, 0.0), 4))
print("Top-5 :", round(test_accs.get(5, 0.0), 4))
print("Top-10:", round(test_accs.get(10, 0.0), 4))


TEST (Gloss Retrieval) Accuracies:
Top-1 : 0.1431
Top-5 : 0.1702
Top-10: 0.1864


In [121]:
# how many test rows dropped due to unseen gloss?
print("Dropped test rows (unseen gloss):", len(df_test) - len(df_test_seen))

# unique gloss counts
print("Unique glosses in prototype:", df_prototype["gloss_norm"].nunique())
print("Unique glosses in test_seen:", df_test_seen["gloss_norm"].nunique())

# distribution: how many prototype videos per gloss (important for choosing M)
gloss_counts = df_prototype["gloss_norm"].value_counts()
print("Prototype videos per gloss: min/median/max =", gloss_counts.min(), gloss_counts.median(), gloss_counts.max())
print("Glosses with >=2 videos:", (gloss_counts >= 2).sum())
print("Glosses with >=3 videos:", (gloss_counts >= 3).sum())


Dropped test rows (unseen gloss): 0
Unique glosses in prototype: 4765
Unique glosses in test_seen: 1363
Prototype videos per gloss: min/median/max = 1 1.0 1
Glosses with >=2 videos: 0
Glosses with >=3 videos: 0


In [124]:
rows = []
for uid, enc_g in zip(df_prototype["uid"].values, df_prototype["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    rows.append((uid, tensor, enc_g))

df_train_gloss = pd.DataFrame(rows, columns=["uid", "I3D_features", "encoded_gloss"])
df_tr_g, df_val_g = train_test_split(df_train_gloss, test_size=0.2, random_state=SEED, shuffle=True)
ds_tr = DFVideoDataset(df_tr_g.rename(columns={"encoded_gloss":"encoded_category"}), label_col="encoded_category")
train_loader = DataLoader(ds_tr, batch_size=8, shuffle=True, collate_fn=collate_fn)
ds_val_support = DFVideoDataset(df_tr_g.rename(columns={"encoded_gloss":"encoded_category"}), label_col="encoded_category")
ds_val_query   = DFVideoDataset(df_val_g.rename(columns={"encoded_gloss":"encoded_category"}), label_col="encoded_category")

val_support_loader = DataLoader(ds_val_support, batch_size=32, shuffle=False, collate_fn=collate_fn)
val_query_loader   = DataLoader(ds_val_query,   batch_size=32, shuffle=False, collate_fn=collate_fn)
num_gloss = df_train_gloss["encoded_gloss"].nunique()  # ~4765
ds_val = DFVideoDataset(df_val_g.rename(columns={"encoded_gloss":"encoded_category"}), label_col="encoded_category")
val_loader = DataLoader(ds_val, batch_size=32, shuffle=False, collate_fn=collate_fn)

num_gloss = df_train_gloss["encoded_gloss"].nunique()

model = train_stage_gloss_ce(
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=num_gloss,
    epochs=30,
    lr=1e-4,
    weight_decay=0.01,
    patience=5,
    device=device
)



Epoch 01 | TrainLoss=8.4832 TrainAcc=0.0000 | ValLoss=8.5027 ValAcc=0.0000


Epoch 02 | TrainLoss=8.4134 TrainAcc=0.0000 | ValLoss=8.9326 ValAcc=0.0000


Epoch 03 | TrainLoss=8.3260 TrainAcc=0.0026 | ValLoss=9.2741 ValAcc=0.0000


Epoch 04 | TrainLoss=8.2683 TrainAcc=0.0131 | ValLoss=9.4944 ValAcc=0.0000


Epoch 05 | TrainLoss=8.2197 TrainAcc=0.0567 | ValLoss=9.6324 ValAcc=0.0000


Epoch 06 | TrainLoss=8.1750 TrainAcc=0.1375 | ValLoss=9.7395 ValAcc=0.0000
Early stopping at epoch 6 (best ValLoss=8.5027 at epoch 1).
Restored best model from epoch 1 (ValLoss=8.5027).


In [149]:
# --- 0) setup
def normalize_gloss(g):
    return " ".join(str(g).strip().upper().split())

df_prototype = df_prototype.copy()
df_test0 = df_test.copy()

df_prototype["gloss_norm"] = df_prototype["gloss"].apply(normalize_gloss)
df_test0["gloss_norm"] = df_test0["gloss"].apply(normalize_gloss)

# --- 1) Category encoder (fit on prototype only)
le = LabelEncoder()
df_prototype["encoded_category"] = le.fit_transform(df_prototype["category"].astype(str))

proto_cats = set(df_prototype["category"].astype(str))
df_test_cat_seen = df_test0[df_test0["category"].astype(str).isin(proto_cats)].copy()
df_test_cat_seen["encoded_category"] = le.transform(df_test_cat_seen["category"].astype(str))

print("Dropped test rows (unseen category):", len(df_test0) - len(df_test_cat_seen))

# --- 2) Gloss encoder (fit on prototype only)
gloss_le = LabelEncoder()
df_prototype["encoded_gloss"] = gloss_le.fit_transform(df_prototype["gloss_norm"])

proto_glosses = set(df_prototype["gloss_norm"])
df_test_seen = df_test_cat_seen[df_test_cat_seen["gloss_norm"].isin(proto_glosses)].copy()
df_test_seen["encoded_gloss"] = gloss_le.transform(df_test_seen["gloss_norm"])

print("Dropped test rows (unseen gloss):", len(df_test_cat_seen) - len(df_test_seen))

# --- 3) sanity: df_test_seen MUST have both columns now
print("df_test_seen cols:", df_test_seen.columns.tolist())
print("has encoded_category?", "encoded_category" in df_test_seen.columns)
print("has encoded_gloss?", "encoded_gloss" in df_test_seen.columns)

# --- Prototype full
proto_rows = []
for uid, cat, gloss in zip(df_prototype["uid"].values,
                          df_prototype["encoded_category"].values,
                          df_prototype["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    proto_rows.append((uid, tensor, cat, gloss))
df_proto_full = pd.DataFrame(proto_rows, columns=["uid","I3D_features","encoded_category","encoded_gloss"])

# --- Test full
test_rows = []
for uid, cat, gloss in zip(df_test_seen["uid"].values,
                          df_test_seen["encoded_category"].values,
                          df_test_seen["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    test_rows.append((uid, tensor, cat, gloss))
df_test_full = pd.DataFrame(test_rows, columns=["uid","I3D_features","encoded_category","encoded_gloss"])

print("PROTO full cols:", df_proto_full.columns.tolist())
print("TEST  full cols:", df_test_full.columns.tolist())
print("PROTO shape:", df_proto_full.shape, "TEST shape:", df_test_full.shape)

@torch.no_grad()
def embed_bank(encoder, loader, device):
    encoder.eval()
    Z = []
    cat = []
    gloss = []
    for x, m, y_cat, y_gloss in loader:
        x, m = x.to(device), m.to(device)
        z = encoder(x, m).cpu()  # (B,d)
        Z.append(z)
        cat.append(y_cat.cpu())
        gloss.append(y_gloss.cpu())
    Z = torch.cat(Z, dim=0)             # (P,d)
    cat = torch.cat(cat, dim=0)         # (P,)
    gloss = torch.cat(gloss, dim=0)     # (P,)
    return Z, cat, gloss

class DFVideoDataset2(torch.utils.data.Dataset):
    """returns x, y_cat, y_gloss"""
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        x = np.array(self.df.loc[idx, "I3D_features"])
        y_cat = int(self.df.loc[idx, "encoded_category"])
        y_gloss = int(self.df.loc[idx, "encoded_gloss"])
        return x, y_cat, y_gloss

def collate_fn2(batch, T_max=51):
    xs = [b[0] for b in batch]
    y_cat = [b[1] for b in batch]
    y_gloss = [b[2] for b in batch]

    B = len(batch)
    padded = np.zeros((B, 1024, T_max), np.float32)
    mask   = np.zeros((B, T_max), np.float32)
    for i, feat in enumerate(xs):
        p, m, _ = pad_features(feat, T_max=T_max)
        padded[i] = p
        mask[i] = m

    return (torch.from_numpy(padded),
            torch.from_numpy(mask),
            torch.tensor(y_cat, dtype=torch.long),
            torch.tensor(y_gloss, dtype=torch.long))
proto_loader2 = DataLoader(DFVideoDataset2(df_proto_full), batch_size=64, shuffle=False, collate_fn=collate_fn2)
test_loader2  = DataLoader(DFVideoDataset2(df_test_full),  batch_size=64, shuffle=False, collate_fn=collate_fn2)

device = "cuda" if torch.cuda.is_available() else "mps"

# prototype bank
Zp, cat_p, gloss_p = embed_bank(model.encoder, proto_loader2, device=device)  # Zp: (P,d), already L2 norm
Zp = Zp.to(device)
cat_p = cat_p.to(device)
gloss_p = gloss_p.to(device)

# map category -> prototype indices
cat_to_idx = {}
for i in range(len(cat_p)):
    c = int(cat_p[i].item())
    cat_to_idx.setdefault(c, []).append(i)
@torch.no_grad()
def gated_gloss_retrieval_eval(model, test_loader2, Zp, cat_p, gloss_p, cat_to_idx, device, topC=3, topk=(1,5,10)):
    model.eval()
    hits = {k: 0 for k in topk}
    total = 0

    for x, m, y_cat_true, y_gloss_true in test_loader2:
        x, m = x.to(device), m.to(device)
        y_gloss_true = y_gloss_true.to(device)

        logits_cat, z = model(x, m)  # logits over category, z normalized
        total += z.size(0)

        topC_cats = logits_cat.topk(topC, dim=1).indices  # (B, topC)

        for i in range(z.size(0)):
            # gather candidate prototype indices from predicted categories
            cand = []
            for c in topC_cats[i].tolist():
                cand.extend(cat_to_idx.get(int(c), []))

            # safety fallback (shouldn't happen)
            if len(cand) == 0:
                cand = list(range(Zp.size(0)))

            cand = torch.tensor(cand, device=device, dtype=torch.long)
            sims = (z[i:i+1] @ Zp[cand].t()).squeeze(0)  # (Ncand,)

            # rank candidates
            # take topK candidates then map to gloss ids
            maxK = max(topk)
            top_idx = sims.topk(min(maxK, sims.numel())).indices  # indices in cand
            pred_gloss = gloss_p[cand[top_idx]]  # (K,)

            true_g = y_gloss_true[i].item()
            for k in topk:
                if true_g in pred_gloss[:k].tolist():
                    hits[k] += 1

    return {k: hits[k]/max(total,1) for k in topk}

accs_gated = gated_gloss_retrieval_eval(
    model, test_loader2,
    Zp, cat_p, gloss_p, cat_to_idx,
    device=device, topC=3, topk=(1,5,10)
)

print("TEST (Gated Gloss Retrieval) Accuracies:")
print("Top-1 :", round(accs_gated.get(1,0), 4))
print("Top-5 :", round(accs_gated.get(5,0), 4))
print("Top-10:", round(accs_gated.get(10,0), 4))


Dropped test rows (unseen category): 1
Dropped test rows (unseen gloss): 0
df_test_seen cols: ['uid', 'gloss', 'duration', 'category', 'gloss_norm', 'encoded_category', 'encoded_gloss']
has encoded_category? True
has encoded_gloss? True
PROTO full cols: ['uid', 'I3D_features', 'encoded_category', 'encoded_gloss']
TEST  full cols: ['uid', 'I3D_features', 'encoded_category', 'encoded_gloss']
PROTO shape: (4765, 4) TEST shape: (2284, 4)
TEST (Gated Gloss Retrieval) Accuracies:
Top-1 : 0.1559
Top-5 : 0.1839
Top-10: 0.1966


In [151]:
for topC in [1, 3, 5]:
    accs_gated = gated_gloss_retrieval_eval(
        model, test_loader2,
        Zp, cat_p, gloss_p, cat_to_idx,
        device=device, topC=topC, topk=(1,5,10)
    )
    print(f"\nTopC={topC}")
    print("Top-1 :", round(accs_gated.get(1,0), 4))
    print("Top-5 :", round(accs_gated.get(5,0), 4))
    print("Top-10:", round(accs_gated.get(10,0), 4))



TopC=1
Top-1 : 0.1589
Top-5 : 0.1887
Top-10: 0.2018

TopC=3
Top-1 : 0.1559
Top-5 : 0.1839
Top-10: 0.1966

TopC=5
Top-1 : 0.1506
Top-5 : 0.1778
Top-10: 0.19


In [152]:
@torch.no_grad()
def soft_gated_gloss_retrieval_eval(model, test_loader2, Zp, cat_p, gloss_p, device,
                                   lam=0.5, topk=(1,5,10)):
    model.eval()
    hits = {k: 0 for k in topk}
    total = 0

    for x, m, y_cat_true, y_gloss_true in test_loader2:
        x, m = x.to(device), m.to(device)
        y_gloss_true = y_gloss_true.to(device)

        logits_cat, z = model(x, m)              # z: (B,d)
        logp = torch.log_softmax(logits_cat, 1)  # (B,57)

        # cosine sims to ALL prototypes
        sims = z @ Zp.t()                        # (B,P)

        # add category prior per prototype
        # cat_p is (P,) with category id for each prototype
        bonus = logp[:, cat_p]                   # (B,P)
        scores = sims + lam * bonus              # (B,P)

        # retrieve by prototype, then map to gloss ids
        maxK = max(topk)
        top_idx = scores.topk(maxK, dim=1).indices  # (B,K)
        pred_gloss = gloss_p[top_idx]               # (B,K)

        total += z.size(0)
        for k in topk:
            hit = (pred_gloss[:, :k] == y_gloss_true.unsqueeze(1)).any(1).sum().item()
            hits[k] += hit

    return {k: hits[k]/max(total,1) for k in topk}


In [153]:
for lam in [0.0, 0.1, 0.25, 0.5, 1.0, 2.0]:
    accs = soft_gated_gloss_retrieval_eval(model, test_loader2, Zp, cat_p, gloss_p, device, lam=lam)
    print(f"\nλ={lam}")
    print("Top-1 :", round(accs.get(1,0), 4))
    print("Top-5 :", round(accs.get(5,0), 4))
    print("Top-10:", round(accs.get(10,0), 4))



λ=0.0
Top-1 : 0.1611
Top-5 : 0.1909
Top-10: 0.204

λ=0.1
Top-1 : 0.1607
Top-5 : 0.1909
Top-10: 0.2049

λ=0.25
Top-1 : 0.1563
Top-5 : 0.187
Top-10: 0.1992

λ=0.5
Top-1 : 0.1235
Top-5 : 0.1686
Top-10: 0.187

λ=1.0
Top-1 : 0.067
Top-5 : 0.1007
Top-10: 0.1195

λ=2.0
Top-1 : 0.0215
Top-5 : 0.0403
Top-10: 0.0495


In [154]:
@torch.no_grad()
def gloss_retrieval_eval_same_impl(model, test_loader2, Zp, gloss_p, device, topk=(1,5,10)):
    model.eval()
    hits = {k: 0 for k in topk}
    total = 0
    for x, m, _, y_gloss_true in test_loader2:
        x, m = x.to(device), m.to(device)
        y_gloss_true = y_gloss_true.to(device)

        _, z = model(x, m)
        sims = z @ Zp.t()   # (B,P)

        maxK = max(topk)
        top_idx = sims.topk(maxK, dim=1).indices
        pred_gloss = gloss_p[top_idx]

        total += z.size(0)
        for k in topk:
            hits[k] += (pred_gloss[:, :k] == y_gloss_true.unsqueeze(1)).any(1).sum().item()

    return {k: hits[k]/max(total,1) for k in topk}


In [155]:
accs_ungated = gloss_retrieval_eval_same_impl(model, test_loader2, Zp, gloss_p, device)
print(accs_ungated)


{1: 0.16112084063047286, 5: 0.19089316987740806, 10: 0.2040280210157618}


In [156]:
# Zp: (P, d) on device, already L2-normalized from your encoder
mu = Zp.mean(dim=0, keepdim=True)             # (1, d)

Zp_center = Zp - mu                           # (P, d)
Zp_center = torch.nn.functional.normalize(Zp_center, p=2, dim=1)  # re-normalize
@torch.no_grad()
def gloss_retrieval_eval_centered(model, test_loader2, Zp_center, mu, gloss_p, device, topk=(1,5,10)):
    model.eval()
    hits = {k: 0 for k in topk}
    total = 0

    for x, m, _, y_gloss_true in test_loader2:
        x, m = x.to(device), m.to(device)
        y_gloss_true = y_gloss_true.to(device)

        _, z = model(x, m)                  # (B,d), L2-normalized
        zc = z - mu                         # center
        zc = torch.nn.functional.normalize(zc, p=2, dim=1)

        sims = zc @ Zp_center.t()           # (B,P)
        maxK = max(topk)
        top_idx = sims.topk(maxK, dim=1).indices
        pred_gloss = gloss_p[top_idx]

        total += z.size(0)
        for k in topk:
            hits[k] += (pred_gloss[:, :k] == y_gloss_true.unsqueeze(1)).any(1).sum().item()

    return {k: hits[k]/max(total,1) for k in topk}
accs_centered = gloss_retrieval_eval_centered(model, test_loader2, Zp_center, mu, gloss_p, device)
print("Centered cosine:", accs_centered)

# (optional) print side-by-side
print("Ungated baseline:", {1:0.16112084063047286, 5:0.19089316987740806, 10:0.2040280210157618})


Centered cosine: {1: 0.1672504378283713, 5: 0.20140105078809106, 10: 0.22154115586690018}
Ungated baseline: {1: 0.16112084063047286, 5: 0.19089316987740806, 10: 0.2040280210157618}


In [173]:
@torch.no_grad()
def retrieval_eval_safe(encoder, query_loader, proto_classes, proto_mat, device, topk=(1,5,10)):
    encoder.eval()
    proto_mat = proto_mat.to(device)
    idx_map = {c:i for i,c in enumerate(proto_classes)}

    topk_hits = {k: 0 for k in topk}
    total = 0

    for x, m, y in query_loader:
        x, m, y = x.to(device), m.to(device), y.to(device)

        keep = torch.tensor([int(t.item()) in idx_map for t in y], device=device)
        if keep.sum().item() == 0:
            continue

        x, m, y = x[keep], m[keep], y[keep]
        z = encoder(x, m)
        sims = z @ proto_mat.t()

        true_idx = torch.tensor([idx_map[int(t.item())] for t in y], device=device)

        for k in topk:
            pred_idx = sims.topk(k, dim=1).indices
            topk_hits[k] += (pred_idx == true_idx.unsqueeze(1)).any(dim=1).sum().item()

        total += z.size(0)

    return {k: topk_hits[k]/max(total,1) for k in topk}, total


In [157]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, z, y):
        """
        z: (B, d) L2-normalized embeddings
        y: (B,) labels (category)
        """
        device = z.device
        B = z.size(0)

        sim = (z @ z.t()) / self.tau              # (B,B)
        sim = sim - sim.max(dim=1, keepdim=True).values  # stability

        y = y.view(-1, 1)
        pos_mask = (y == y.t()).float().to(device)       # (B,B)
        self_mask = torch.eye(B, device=device)
        pos_mask = pos_mask * (1 - self_mask)            # remove self positives

        exp_sim = torch.exp(sim) * (1 - self_mask)       # remove self from denom
        denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

        log_prob = sim - torch.log(denom)                # (B,B)

        # average over positives per anchor
        pos_count = pos_mask.sum(dim=1) + 1e-12
        loss = -(pos_mask * log_prob).sum(dim=1) / pos_count
        return loss.mean()


In [158]:
from tqdm.auto import tqdm
import copy

def train_stage2_supcon(
    model,                   # EncoderWithHead from stage1 (we will fine-tune model.encoder)
    train_loader,
    val_support_loader,      # your k-shot category support loader
    val_query_loader,
    epochs=20,
    lr=3e-5,
    weight_decay=0.05,
    temperature=0.07,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
):
    model = model.to(device)
    model.train()

    # fine-tune encoder params (optionally include head, but not needed)
    opt = torch.optim.AdamW(model.encoder.parameters(), lr=lr, weight_decay=weight_decay)
    supcon = SupConLoss(temperature=temperature)

    best_top1 = -1.0
    best_epoch = -1
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs+1):
        model.train()
        total_loss = 0.0
        total_seen = 0

        pbar = tqdm(train_loader, desc=f"Stage2 {ep}/{epochs}", leave=False)
        for x, m, y in pbar:      # y = encoded_category (same as stage1 training)
            x, m, y = x.to(device), m.to(device), y.to(device)

            z = model.encoder(x, m)           # (B,d) already normalized in your encoder
            loss = supcon(z, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_seen += bs
            pbar.set_postfix(loss=total_loss/max(total_seen,1))

        train_loss = total_loss / max(total_seen, 1)

        # same retrieval validation you already trust (category retrieval on val)
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs = retrieval_eval(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)
        top1 = float(val_accs.get(1, 0.0))

        print(f"[Stage2] Epoch {ep:02d} | SupConLoss={train_loss:.4f} | ValRetr Top1={top1:.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f}")

        # early stopping on val retrieval top1
        if top1 > best_top1 + min_delta:
            best_top1 = top1
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"[Stage2] Early stopping at epoch {ep} (best Top1={best_top1:.4f} at epoch {best_epoch}).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[Stage2] Restored best model from epoch {best_epoch} (Top1={best_top1:.4f}).")

    return model


In [171]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import copy

class SupConLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.tau = temperature

    def forward(self, z, y):
        device = z.device
        B = z.size(0)

        sim = (z @ z.t()) / self.tau
        sim = sim - sim.max(dim=1, keepdim=True).values  # stability

        y = y.view(-1, 1)
        pos_mask = (y == y.t()).float().to(device)

        self_mask = torch.eye(B, device=device)
        pos_mask = pos_mask * (1 - self_mask)

        exp_sim = torch.exp(sim) * (1 - self_mask)
        denom = exp_sim.sum(dim=1, keepdim=True) + 1e-12

        log_prob = sim - torch.log(denom)

        pos_count = pos_mask.sum(dim=1) + 1e-12
        loss = -(pos_mask * log_prob).sum(dim=1) / pos_count
        return loss.mean()


def train_stage2_supcon(
    model,
    train_loader,
    val_support_loader,
    val_query_loader,
    epochs=20,
    lr=3e-5,
    weight_decay=0.05,
    temperature=0.07,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
):
    model = model.to(device)

    # fine-tune encoder only
    opt = torch.optim.AdamW(model.encoder.parameters(), lr=lr, weight_decay=weight_decay)
    supcon = SupConLoss(temperature=temperature)

    best_top1 = -1.0
    best_epoch = -1
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_seen = 0

        pbar = tqdm(train_loader, desc=f"Stage2 {ep}/{epochs}", leave=False)
        for x, m, y in pbar:
            x, m, y = x.to(device), m.to(device), y.to(device)

            z = model.encoder(x, m)   # (B,d) already normalized
            loss = supcon(z, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_seen += bs
            pbar.set_postfix(loss=total_loss / max(total_seen, 1))

        train_loss = total_loss / max(total_seen, 1)

        # validation retrieval on categories (same eval as stage1)
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs, kept = retrieval_eval_safe(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)
        print(f"[Stage2] Epoch {ep:02d} | SupConLoss={train_loss:.4f} | "
        f"ValRetr Top1={val_accs.get(1,0):.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f} | "
        f"ValKept={kept}")


        top1 = float(val_accs.get(1, 0.0))

        print(
            f"[Stage2] Epoch {ep:02d} | SupConLoss={train_loss:.4f} | "
            f"ValRetr Top1={val_accs.get(1,0):.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f}"
        )

        # early stopping
        if top1 > best_top1 + min_delta:
            best_top1 = top1
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"[Stage2] Early stopping at epoch {ep} (best Top1={best_top1:.4f} at epoch {best_epoch}).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[Stage2] Restored best model from epoch {best_epoch} (Top1={best_top1:.4f}).")

    return model


In [165]:
device = "cuda" if torch.cuda.is_available() else "mps"

# ---- Stage-2 fine-tune (SupCon on category labels) ----
model = train_stage2_supcon(
    model=model,                         # your stage-1 trained model
    train_loader=train_loader,           # category train loader
    val_support_loader=val_support_loader,
    val_query_loader=val_query_loader,
    epochs=20,
    lr=3e-5,
    weight_decay=0.05,
    temperature=0.07,
    patience=5,
    min_delta=1e-4,
    device=device
)


[Stage2] Epoch 01 | SupConLoss=0.0000 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 02 | SupConLoss=0.0000 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 03 | SupConLoss=0.0000 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 04 | SupConLoss=0.0000 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 05 | SupConLoss=0.0000 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 06 | SupConLoss=0.0000 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000
[Stage2] Early stopping at epoch 6 (best Top1=0.0000 at epoch 1).
[Stage2] Restored best model from epoch 1 (Top1=0.0000).


In [167]:
import numpy as np
import torch
from torch.utils.data import Sampler

class EpisodicBatchSampler(Sampler):
    """
    Creates batches with n_classes per batch and n_samples per class.
    Ensures positives exist for SupCon.
    """
    def __init__(self, labels, n_classes=4, n_samples=4, seed=1990):
        self.labels = np.array(labels)
        self.n_classes = n_classes
        self.n_samples = n_samples
        self.rng = np.random.default_rng(seed)

        self.classes = np.unique(self.labels)
        self.idxs_by_class = {c: np.where(self.labels == c)[0] for c in self.classes}

        # how many batches per epoch (rough)
        self.batch_size = n_classes * n_samples
        self.num_batches = len(self.labels) // self.batch_size

    def __iter__(self):
        for _ in range(self.num_batches):
            chosen_classes = self.rng.choice(self.classes, size=self.n_classes, replace=False)
            batch = []
            for c in chosen_classes:
                idxs = self.idxs_by_class[c]
                # sample with replacement if class is small
                replace = len(idxs) < self.n_samples
                picked = self.rng.choice(idxs, size=self.n_samples, replace=replace)
                batch.extend(picked.tolist())
            self.rng.shuffle(batch)
            yield batch

    def __len__(self):
        return self.num_batches


In [168]:
labels_tr = ds_tr.df["encoded_category"].values  # same label col you used
batch_sampler = EpisodicBatchSampler(labels_tr, n_classes=4, n_samples=4, seed=SEED)

train_loader_supcon = DataLoader(
    ds_tr,
    batch_sampler=batch_sampler,
    collate_fn=collate_fn
)

# sanity: check duplicates exist
x, m, y = next(iter(train_loader_supcon))
print("batch size:", y.shape[0], "unique classes:", y.unique().numel())


batch size: 16 unique classes: 4


In [174]:
model = train_stage2_supcon(
    model=model,
    train_loader=train_loader_supcon,   # <-- use new one
    val_support_loader=val_support_loader,
    val_query_loader=val_query_loader,
    epochs=20,
    lr=3e-5,
    weight_decay=0.05,
    temperature=0.07,
    patience=5,
    min_delta=1e-4,
    device=device
)


[Stage2] Epoch 01 | SupConLoss=1.0989 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000 | ValKept=0
[Stage2] Epoch 01 | SupConLoss=1.0989 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 02 | SupConLoss=1.0988 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000 | ValKept=0
[Stage2] Epoch 02 | SupConLoss=1.0988 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 03 | SupConLoss=1.0989 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000 | ValKept=0
[Stage2] Epoch 03 | SupConLoss=1.0989 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 04 | SupConLoss=1.0988 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000 | ValKept=0
[Stage2] Epoch 04 | SupConLoss=1.0988 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 05 | SupConLoss=1.0987 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000 | ValKept=0
[Stage2] Epoch 05 | SupConLoss=1.0987 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000


[Stage2] Epoch 06 | SupConLoss=1.0987 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000 | ValKept=0
[Stage2] Epoch 06 | SupConLoss=1.0987 | ValRetr Top1=0.0000 Top5=0.0000 Top10=0.0000
[Stage2] Early stopping at epoch 6 (best Top1=0.0000 at epoch 1).
[Stage2] Restored best model from epoch 1 (Top1=0.0000).


In [175]:
k = 5
cols = ["uid", "I3D_features", "encoded_category"]

# 1) Keep only classes with >= k+1 samples so support+query are possible
vc = df_val["encoded_category"].value_counts()
eligible = vc[vc >= (k + 1)].index
df_val_k = df_val[df_val["encoded_category"].isin(eligible)].copy()

# 2) Sample k support per class (deterministic)
df_val_support = (
    df_val_k[cols]
    .groupby("encoded_category", group_keys=False)
    .sample(n=k, random_state=SEED)
)

# 3) Query = remaining rows
df_val_query = df_val_k.drop(index=df_val_support.index)

# 4) Reset indices
df_val_support = df_val_support.reset_index(drop=True)
df_val_query   = df_val_query.reset_index(drop=True)

# 5) HARD CHECKS (must pass)
support_classes = set(df_val_support["encoded_category"].unique())
query_classes   = set(df_val_query["encoded_category"].unique())

print("support rows:", len(df_val_support), "support classes:", len(support_classes))
print("query rows:", len(df_val_query), "query classes:", len(query_classes))
print("query minus support:", len(query_classes - support_classes))
print("support minus query:", len(support_classes - query_classes))

assert len(query_classes - support_classes) == 0, "Mismatch: query has classes not in support!"

# 6) Build datasets/loaders
ds_val_support = DFVideoDataset(df_val_support, label_col="encoded_category")
ds_val_query   = DFVideoDataset(df_val_query,   label_col="encoded_category")

val_support_loader = DataLoader(ds_val_support, batch_size=64, shuffle=False, collate_fn=collate_fn)
val_query_loader   = DataLoader(ds_val_query,   batch_size=64, shuffle=False, collate_fn=collate_fn)


support rows: 225 support classes: 45
query rows: 685 query classes: 45
query minus support: 0
support minus query: 0


In [176]:
model = train_stage2_supcon(
    model=model,
    train_loader=train_loader_supcon,     # the episodic sampler loader
    val_support_loader=val_support_loader,
    val_query_loader=val_query_loader,
    epochs=20,
    lr=3e-5,
    weight_decay=0.05,
    temperature=0.07,
    patience=5,
    min_delta=1e-4,
    device=device
)


[Stage2] Epoch 01 | SupConLoss=1.0989 | ValRetr Top1=0.0832 Top5=0.2292 Top10=0.3635 | ValKept=685
[Stage2] Epoch 01 | SupConLoss=1.0989 | ValRetr Top1=0.0832 Top5=0.2292 Top10=0.3635


[Stage2] Epoch 02 | SupConLoss=1.0988 | ValRetr Top1=0.0861 Top5=0.2292 Top10=0.3635 | ValKept=685
[Stage2] Epoch 02 | SupConLoss=1.0988 | ValRetr Top1=0.0861 Top5=0.2292 Top10=0.3635


[Stage2] Epoch 03 | SupConLoss=1.0987 | ValRetr Top1=0.0847 Top5=0.2307 Top10=0.3679 | ValKept=685
[Stage2] Epoch 03 | SupConLoss=1.0987 | ValRetr Top1=0.0847 Top5=0.2307 Top10=0.3679


[Stage2] Epoch 04 | SupConLoss=1.0988 | ValRetr Top1=0.0759 Top5=0.2161 Top10=0.3679 | ValKept=685
[Stage2] Epoch 04 | SupConLoss=1.0988 | ValRetr Top1=0.0759 Top5=0.2161 Top10=0.3679


[Stage2] Epoch 05 | SupConLoss=1.0988 | ValRetr Top1=0.0730 Top5=0.2204 Top10=0.3708 | ValKept=685
[Stage2] Epoch 05 | SupConLoss=1.0988 | ValRetr Top1=0.0730 Top5=0.2204 Top10=0.3708


[Stage2] Epoch 06 | SupConLoss=1.0987 | ValRetr Top1=0.0715 Top5=0.2234 Top10=0.3766 | ValKept=685
[Stage2] Epoch 06 | SupConLoss=1.0987 | ValRetr Top1=0.0715 Top5=0.2234 Top10=0.3766


[Stage2] Epoch 07 | SupConLoss=1.0988 | ValRetr Top1=0.0774 Top5=0.2190 Top10=0.3591 | ValKept=685
[Stage2] Epoch 07 | SupConLoss=1.0988 | ValRetr Top1=0.0774 Top5=0.2190 Top10=0.3591
[Stage2] Early stopping at epoch 7 (best Top1=0.0861 at epoch 2).
[Stage2] Restored best model from epoch 2 (Top1=0.0861).


In [177]:
device = "cuda" if torch.cuda.is_available() else "mps"

# loaders (you already had these in step1)
proto_loader2 = DataLoader(DFVideoDataset2(df_proto_full), batch_size=64, shuffle=False, collate_fn=collate_fn2)
test_loader2  = DataLoader(DFVideoDataset2(df_test_full),  batch_size=64, shuffle=False, collate_fn=collate_fn2)

@torch.no_grad()
def embed_proto_bank(model, proto_loader2, device):
    model.eval()
    Z_list, cat_list, gloss_list = [], [], []
    for x, m, y_cat, y_gloss in tqdm(proto_loader2, desc="Embedding prototypes", leave=False):
        x, m = x.to(device), m.to(device)
        _, z = model(x, m)
        Z_list.append(z.detach().cpu())
        cat_list.append(y_cat.cpu())
        gloss_list.append(y_gloss.cpu())
    Zp = torch.cat(Z_list, dim=0).to(device)
    cat_p = torch.cat(cat_list, dim=0).to(device)
    gloss_p = torch.cat(gloss_list, dim=0).to(device)
    return Zp, cat_p, gloss_p

Zp, cat_p, gloss_p = embed_proto_bank(model, proto_loader2, device)

mu = Zp.mean(dim=0, keepdim=True)
Zp_center = F.normalize(Zp - mu, p=2, dim=1)

accs_centered_stage2 = gloss_retrieval_eval_centered(
    model=model,
    test_loader2=test_loader2,
    Zp_center=Zp_center,
    mu=mu,
    gloss_p=gloss_p,
    device=device,
    topk=(1,5,10)
)

print("TEST (Centered cosine) after Stage-2 SupCon:")
print(accs_centered_stage2)


TEST (Centered cosine) after Stage-2 SupCon:
{1: 0.16943957968476359, 5: 0.20490367775831875, 10: 0.22898423817863398}


In [178]:
def train_stage2_ce_supcon(
    model,
    train_loader_supcon,
    val_support_loader,
    val_query_loader,
    epochs=15,
    lr=1e-5,
    weight_decay=0.05,
    temperature=0.07,
    alpha=0.1,               # weight for SupCon
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    supcon = SupConLoss(temperature=temperature)
    ce = nn.CrossEntropyLoss()

    best_top1, best_epoch, best_state, bad_epochs = -1.0, -1, None, 0

    for ep in range(1, epochs+1):
        model.train()
        total_loss, total_seen = 0.0, 0

        pbar = tqdm(train_loader_supcon, desc=f"Stage2(CE+SupCon) {ep}/{epochs}", leave=False)
        for x, m, y in pbar:
            x, m, y = x.to(device), m.to(device), y.to(device)

            logits, z = model(x, m)
            loss_ce = ce(logits, y)

            # skip if no positives (should not happen with episodic sampler)
            if torch.unique(y).numel() == y.numel():
                loss_sc = torch.tensor(0.0, device=device)
            else:
                loss_sc = supcon(z, y)

            loss = loss_ce + alpha * loss_sc

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_seen += bs
            pbar.set_postfix(loss=total_loss / max(total_seen,1))

        # validation
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs, kept = retrieval_eval_safe(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)

        top1 = float(val_accs.get(1,0.0))
        print(f"[Stage2 CE+SupCon] Epoch {ep:02d} | Loss={total_loss/max(total_seen,1):.4f} | "
              f"ValTop1={top1:.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f} | Kept={kept}")

        if top1 > best_top1 + min_delta:
            best_top1, best_epoch = top1, ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stop at {ep}, best {best_top1:.4f} at {best_epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored epoch {best_epoch} (Top1={best_top1:.4f})")

    return model


In [179]:
model = train_stage2_ce_supcon(
    model=model,  # start from best Stage-1 checkpoint ideally
    train_loader_supcon=train_loader_supcon,
    val_support_loader=val_support_loader,
    val_query_loader=val_query_loader,
    epochs=15,
    lr=1e-5,
    alpha=0.1,
    temperature=0.07,
    weight_decay=0.05,
    patience=5,
    device=device
)


[Stage2 CE+SupCon] Epoch 01 | Loss=8.5399 | ValTop1=0.0861 Top5=0.2234 Top10=0.3737 | Kept=685


[Stage2 CE+SupCon] Epoch 02 | Loss=8.5343 | ValTop1=0.0803 Top5=0.2204 Top10=0.3591 | Kept=685


[Stage2 CE+SupCon] Epoch 03 | Loss=8.5335 | ValTop1=0.0759 Top5=0.2234 Top10=0.3693 | Kept=685


[Stage2 CE+SupCon] Epoch 04 | Loss=8.5329 | ValTop1=0.0745 Top5=0.2219 Top10=0.3591 | Kept=685


[Stage2 CE+SupCon] Epoch 05 | Loss=8.5286 | ValTop1=0.0701 Top5=0.2161 Top10=0.3620 | Kept=685


[Stage2 CE+SupCon] Epoch 06 | Loss=8.5273 | ValTop1=0.0701 Top5=0.2190 Top10=0.3562 | Kept=685
Early stop at 6, best 0.0861 at 1
Restored epoch 1 (Top1=0.0861)


In [193]:
# ---------- CLEAN REBUILD: category training ----------
SEED = 1990

df_prototype = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/prototype.csv")
df_test      = pd.read_csv("/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/splits/test.csv")
df_features  = pd.read_pickle("/Users/lokeshkumar/eMasters/EE964_Projects/data/cislr/features/I3D_features.pkl")

# Encode CATEGORY only
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

cat_le = LabelEncoder()
df_prototype["encoded_category"] = cat_le.fit_transform(df_prototype["category"].astype(str))

# Build df_train_cat with features + category labels
rows = []
for uid, ycat in zip(df_prototype["uid"].astype(str).values, df_prototype["encoded_category"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    rows.append((uid, tensor, int(ycat)))

df_train_cat = pd.DataFrame(rows, columns=["uid", "I3D_features", "encoded_category"])

# Drop singleton categories (need >=2 for stratify and for val split)
counts = df_train_cat["encoded_category"].value_counts()
ok = counts[counts >= 2].index
df_ok = df_train_cat[df_train_cat["encoded_category"].isin(ok)].copy()

df_tr, df_val = train_test_split(
    df_ok, test_size=0.2, random_state=SEED, stratify=df_ok["encoded_category"]
)

print("Category classes (train ok):", df_tr["encoded_category"].nunique())
print("y range:", df_tr["encoded_category"].min(), df_tr["encoded_category"].max())


Category classes (train ok): 57
y range: 0 57


In [194]:
# ---------- BUILD LOADERS ----------
k = 5
cols = ["uid", "I3D_features", "encoded_category"]

vc = df_val["encoded_category"].value_counts()
eligible = vc[vc >= (k + 1)].index
df_val_k = df_val[df_val["encoded_category"].isin(eligible)].copy()

df_val_support = (
    df_val_k[cols]
    .groupby("encoded_category", group_keys=False)
    .sample(n=k, random_state=SEED)
)
df_val_query = df_val_k.drop(index=df_val_support.index)

df_val_support = df_val_support.reset_index(drop=True)
df_val_query   = df_val_query.reset_index(drop=True)

ds_tr = DFVideoDataset(df_tr, label_col="encoded_category")
ds_val_support = DFVideoDataset(df_val_support, label_col="encoded_category")
ds_val_query   = DFVideoDataset(df_val_query,   label_col="encoded_category")

train_loader = DataLoader(ds_tr, batch_size=8, shuffle=True,  collate_fn=collate_fn)
val_support_loader = DataLoader(ds_val_support, batch_size=32, shuffle=False, collate_fn=collate_fn)
val_query_loader   = DataLoader(ds_val_query,   batch_size=32, shuffle=False, collate_fn=collate_fn)

num_classes = df_tr["encoded_category"].nunique()
print("num_classes:", num_classes)


num_classes: 57


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MaskedAttnEncoder(nn.Module):
    """
    Input:
        x: (B, 1024, T)
        mask: (B, T) float {0,1}
    Output:
        z: (B, d) L2-normalized
    """
    def __init__(self, d=256, use_layernorm=True, dropout=0.05):
        super().__init__()
        self.d = d
        self.ln = nn.LayerNorm(1024) if use_layernorm else None

        self.proj = nn.Linear(1024, d)
        self.attn = nn.Linear(d, 1)

        # NEW: pooling fusion head (2d -> d)
        self.pool_fuse = nn.Linear(2 * d, d)

        self.drop = nn.Dropout(p=dropout)

    def masked_mean(self, h, mask):
        # h: (B, T, d), mask: (B, T)
        w = mask.unsqueeze(-1)  # (B,T,1)
        s = (h * w).sum(dim=1)  # (B,d)
        denom = w.sum(dim=1).clamp(min=1e-6)  # (B,1)
        return s / denom

    def masked_max(self, h, mask):
        # set padded positions to -inf before max
        neg_inf = torch.finfo(h.dtype).min
        h2 = h.masked_fill(mask.unsqueeze(-1) == 0, neg_inf)
        return h2.max(dim=1).values

    def forward(self, x, mask):
        # x: (B,1024,T) -> (B,T,1024)
        x = x.transpose(1, 2)

        if self.ln is not None:
            x = self.ln(x)

        h = self.proj(x)           # (B,T,d)
        h = self.drop(h)

        # attention weighted sum (your existing idea)
        scores = self.attn(torch.tanh(h)).squeeze(-1)      # (B,T)
        scores = scores.masked_fill(mask == 0, -1e9)
        alpha = F.softmax(scores, dim=1)                   # (B,T)
        z_attn = torch.sum(h * alpha.unsqueeze(-1), dim=1) # (B,d)

        # NEW: mean+max pooling summary
        z_mean = self.masked_mean(h, mask)  # (B,d)
        z_max  = self.masked_max(h, mask)   # (B,d)

        z_pool = torch.cat([z_mean, z_max], dim=-1)        # (B,2d)
        z_pool = self.pool_fuse(z_pool)                    # (B,d)

        # combine (simple average). You can later try weighted sum.
        z = 0.5 * z_attn + 0.5 * z_pool

        z = F.normalize(z, p=2, dim=-1)
        return z
class EncoderWithHead(nn.Module):
    def __init__(self, num_classes, d=256, dropout=0.05):
        super().__init__()
        self.encoder = MaskedAttnEncoder(d=d, dropout=dropout)
        self.head = nn.Linear(d, num_classes)

    def forward(self, x, mask):
        z = self.encoder(x, mask)      # (B, d)
        logits = self.head(z)          # (B, num_classes)
        return logits, z


In [196]:
# -------------------------
# Stage-1 training (CE) with a one-time sanity debug print
# Paste this whole cell and run.
# -------------------------
from tqdm.auto import tqdm
import copy
import torch
import torch.nn as nn

def train_stage1(
    train_loader,
    val_support_loader,
    val_query_loader,
    num_classes,
    epochs=30,
    lr=1e-4,
    weight_decay=0.05,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps",
    debug_first_batch=True,
):
    model = EncoderWithHead(num_classes=num_classes, d=256).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_top1 = -1.0
    best_epoch = -1
    best_state = None
    bad_epochs = 0

    did_debug = False  # <-- runs sanity print exactly once

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_seen = 0

        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)

        for bidx, (x, m, y) in enumerate(pbar):
            x = x.to(device)
            m = m.to(device)
            y = y.to(device)

            logits, z = model(x, m)
            loss = criterion(logits, y)

            # ---- SANITY DEBUG (first batch only) ----
            if debug_first_batch and (not did_debug) and (ep == 1) and (bidx == 0):
                print("\n=== SANITY CHECK (first batch) ===")
                print("num_classes:", num_classes)
                print("x:", tuple(x.shape), x.dtype)
                print("m:", tuple(m.shape), m.dtype, "mask_sum[0]:", float(m[0].sum().item()))
                print("y:", tuple(y.shape), y.dtype, "min/max:", int(y.min().item()), int(y.max().item()))
                print("logits:", tuple(logits.shape), logits.dtype, "logits[0,:5]:", logits[0, :5].detach().cpu())
                print("z:", tuple(z.shape), z.dtype, "z_norm[0]:", float(z[0].norm().item()))
                assert logits.ndim == 2, f"logits.ndim={logits.ndim}"
                assert logits.size(1) == num_classes, f"logits.size(1)={logits.size(1)} != num_classes={num_classes}"
                assert y.dtype == torch.long, f"y.dtype={y.dtype} (must be torch.long for CE)"
                did_debug = True
                print("✅ Sanity checks passed.\n")

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_seen += bs
            total_correct += (logits.argmax(dim=1) == y).sum().item()

            pbar.set_postfix(
                loss=total_loss / max(total_seen, 1),
                acc=total_correct / max(total_seen, 1),
            )

        train_loss = total_loss / max(total_seen, 1)
        train_acc  = total_correct / max(total_seen, 1)

        # retrieval validation (category)
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs = retrieval_eval(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)
        top1 = float(val_accs.get(1, 0.0))

        print(
            f"Epoch {ep:02d} | "
            f"TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | "
            f"ValRetr Top1={val_accs.get(1,0):.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f}"
        )

        # early stopping on Val Top1
        if top1 > best_top1 + min_delta:
            best_top1 = top1
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"Early stopping at epoch {ep} (best Top1={best_top1:.4f} at epoch {best_epoch}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} (Top1={best_top1:.4f}).")

    return model


# ---- Call example ----
num_classes = df_train["encoded_category"].nunique()
device = "cuda" if torch.cuda.is_available() else "mps"

model = train_stage1(
    train_loader=train_loader,
    val_support_loader=val_support_loader,
    val_query_loader=val_query_loader,
    num_classes=num_classes,
    epochs=3,              # keep small for debugging first
    lr=1e-4,
    weight_decay=0.05,
    patience=5,
    min_delta=1e-4,
    device=device,
    debug_first_batch=True
)


Epoch 1/3:   0%|          | 0/477 [00:00<?, ?it/s]

Epoch 1/3:   4%|▍         | 19/477 [00:00<00:04, 99.25it/s, acc=0.0375, loss=4.04]


=== SANITY CHECK (first batch) ===
num_classes: 58
x: (8, 1024, 51) torch.float32
m: (8, 51) torch.float32 mask_sum[0]: 11.0
y: (8,) torch.int64 min/max: 5 53
logits: (8, 58) torch.float32 logits[0,:5]: tensor([-0.0718, -0.0007, -0.0540,  0.0912, -0.0750])
z: (8, 256) torch.float32 z_norm[0]: 0.9999999403953552
✅ Sanity checks passed.



Epoch 01 | TrainLoss=3.9238 TrainAcc=0.1113 | ValRetr Top1=0.1036 Top5=0.2555 Top10=0.3679


Epoch 02 | TrainLoss=3.7925 TrainAcc=0.1525 | ValRetr Top1=0.1226 Top5=0.2774 Top10=0.3985


Epoch 03 | TrainLoss=3.7023 TrainAcc=0.1671 | ValRetr Top1=0.1518 Top5=0.2978 Top10=0.4263
Restored best model from epoch 3 (Top1=0.1518).


In [197]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import copy

# -------------------------
# SupCon loss (single-view, in-batch)
# -------------------------
def supcon_loss(z, y, temperature=0.07, eps=1e-8):
    """
    z: (B, d) L2-normalized embeddings
    y: (B,) int64 labels
    Returns scalar loss.
    """
    B = z.size(0)
    device = z.device

    # cosine sim since z normalized
    sim = (z @ z.t()) / temperature  # (B,B)

    # mask out self
    logits_mask = torch.ones((B, B), device=device) - torch.eye(B, device=device)

    # positives mask (same label, not self)
    y = y.view(-1, 1)
    pos_mask = (y == y.t()).float() * logits_mask  # (B,B)

    # log-softmax over j (excluding i)
    sim = sim - sim.max(dim=1, keepdim=True).values  # stability
    exp_sim = torch.exp(sim) * logits_mask
    log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + eps)

    # for each i: average log_prob over positives
    pos_count = pos_mask.sum(dim=1)  # (B,)
    # keep only anchors that actually have positives in-batch
    valid = pos_count > 0

    if valid.sum() == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)

    mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / (pos_count + eps)
    loss = -mean_log_prob_pos[valid].mean()
    return loss


# -------------------------
# Stage-2 training (SupCon)
# -------------------------
def train_stage2_supcon(
    model,  # EncoderWithHead from stage1
    train_loader,
    val_support_loader,
    val_query_loader,
    epochs=20,
    lr=3e-5,
    weight_decay=0.05,
    temperature=0.07,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
):
    model = model.to(device)

    # We fine-tune ENCODER only (recommended); head can drift otherwise.
    for p in model.head.parameters():
        p.requires_grad = False

    opt = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )

    best_top1 = -1.0
    best_epoch = -1
    best_state = None
    bad_epochs = 0

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_seen = 0

        pbar = tqdm(train_loader, desc=f"[Stage2] Epoch {ep}/{epochs}", leave=False)

        for x, m, y in pbar:
            x = x.to(device)
            m = m.to(device)
            y = y.to(device)

            # embeddings
            z = model.encoder(x, m)  # (B,d) already L2-normalized in your encoder

            loss = supcon_loss(z, y, temperature=temperature)

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += float(loss.item()) * bs
            total_seen += bs

            pbar.set_postfix(loss=total_loss / max(total_seen, 1))

        train_loss = total_loss / max(total_seen, 1)

        # ---- Validation retrieval (categories) ----
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs = retrieval_eval(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)
        top1 = float(val_accs.get(1, 0.0))

        print(
            f"[Stage2] Epoch {ep:02d} | SupConLoss={train_loss:.4f} | "
            f"ValRetr Top1={val_accs.get(1,0):.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f}"
        )

        # Early stopping on Top1
        if top1 > best_top1 + min_delta:
            best_top1 = top1
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(f"[Stage2] Early stopping at epoch {ep} (best Top1={best_top1:.4f} at epoch {best_epoch}).")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[Stage2] Restored best model from epoch {best_epoch} (Top1={best_top1:.4f}).")

    return model


# -------------------------
# RUN Stage-2
# -------------------------
device = "cuda" if torch.cuda.is_available() else "mps"

model = train_stage2_supcon(
    model=model,
    train_loader=train_loader,
    val_support_loader=val_support_loader,
    val_query_loader=val_query_loader,
    epochs=20,
    lr=3e-5,
    weight_decay=0.05,
    temperature=0.07,
    patience=5,
    min_delta=1e-4,
    device=device
)


[Stage2] Epoch 01 | SupConLoss=1.3030 | ValRetr Top1=0.1328 Top5=0.2788 Top10=0.4131


[Stage2] Epoch 02 | SupConLoss=1.0894 | ValRetr Top1=0.1328 Top5=0.2876 Top10=0.4190


[Stage2] Epoch 03 | SupConLoss=1.0719 | ValRetr Top1=0.1343 Top5=0.2847 Top10=0.4058


[Stage2] Epoch 04 | SupConLoss=1.1168 | ValRetr Top1=0.1299 Top5=0.2832 Top10=0.4117


[Stage2] Epoch 05 | SupConLoss=1.1462 | ValRetr Top1=0.1255 Top5=0.2905 Top10=0.4117


[Stage2] Epoch 06 | SupConLoss=1.1314 | ValRetr Top1=0.1270 Top5=0.2788 Top10=0.4088


[Stage2] Epoch 07 | SupConLoss=1.1259 | ValRetr Top1=0.1241 Top5=0.2715 Top10=0.4219


[Stage2] Epoch 08 | SupConLoss=1.0986 | ValRetr Top1=0.1241 Top5=0.2715 Top10=0.4131
[Stage2] Early stopping at epoch 8 (best Top1=0.1343 at epoch 3).
[Stage2] Restored best model from epoch 3 (Top1=0.1343).


In [198]:
# =========================
# Attentive Statistics Pooling Encoder (mean + std)
# Drop-in replacement for MaskedAttnEncoder
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F

class AttentiveStatsEncoder(nn.Module):
    """
    Input:
        x:    (B, 1024, T)
        mask: (B, T) float {0,1}
    Output:
        z: (B, d) L2-normalized
    """
    def __init__(self, d=256, use_layernorm=True, dropout=0.05, eps=1e-5):
        super().__init__()
        self.d = d
        self.eps = eps

        self.ln = nn.LayerNorm(1024) if use_layernorm else None

        # per-timestep projection: 1024 -> d
        self.proj = nn.Linear(1024, d)
        self.drop = nn.Dropout(p=dropout)

        # attention scorer: d -> 1
        self.attn = nn.Linear(d, 1)

        # stats vector is 2d (mean+std) -> d
        self.out = nn.Linear(2 * d, d)

    def forward(self, x, mask):
        # x: (B, 1024, T) -> (B, T, 1024)
        x = x.transpose(1, 2)  # (B, T, 1024)

        if self.ln is not None:
            x = self.ln(x)

        h = self.proj(x)       # (B, T, d)
        h = self.drop(h)

        # attention scores (B, T)
        scores = self.attn(torch.tanh(h)).squeeze(-1)

        # mask out padding
        scores = scores.masked_fill(mask == 0, -1e9)

        # alpha (B, T)
        alpha = F.softmax(scores, dim=1)

        # ---- weighted mean ----
        # (B, d)
        mu = torch.sum(h * alpha.unsqueeze(-1), dim=1)

        # ---- weighted variance / std ----
        # var = sum(alpha * (h - mu)^2)
        diff = h - mu.unsqueeze(1)                       # (B, T, d)
        var = torch.sum(alpha.unsqueeze(-1) * diff**2, dim=1)  # (B, d)
        std = torch.sqrt(var + self.eps)                 # (B, d)

        # concat stats -> (B, 2d) -> (B, d)
        stats = torch.cat([mu, std], dim=-1)
        z = self.out(stats)

        # L2 normalize
        z = F.normalize(z, p=2, dim=-1)
        return z

class EncoderWithHead(nn.Module):
    def __init__(self, num_classes, d=256, dropout=0.05):
        super().__init__()
        self.encoder = AttentiveStatsEncoder(d=d, use_layernorm=True, dropout=dropout)
        self.head = nn.Linear(d, num_classes)

    def forward(self, x, mask):
        z = self.encoder(x, mask)
        logits = self.head(z)
        return logits, z


In [199]:
# =========================
# Stage-1 training start code (same as your current style)
# Requires: compute_class_prototypes, retrieval_eval already defined
# =========================
from tqdm.auto import tqdm
import copy

def train_stage1(
    train_loader,
    val_support_loader,
    val_query_loader,
    num_classes,
    d=256,
    dropout=0.05,
    epochs=30,
    lr=1e-4,
    weight_decay=0.05,
    patience=5,
    min_delta=1e-4,
    device="cuda" if torch.cuda.is_available() else "mps"
):
    model = EncoderWithHead(num_classes=num_classes, d=d, dropout=dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_top1, best_epoch, best_state = -1.0, -1, None
    bad_epochs = 0
    did_sanity = False

    for ep in range(1, epochs + 1):
        model.train()
        total_loss, total_correct, total_seen = 0.0, 0, 0

        pbar = tqdm(train_loader, desc=f"Epoch {ep}/{epochs}", leave=False)

        for x, m, y in pbar:
            x = x.to(device)
            m = m.to(device)
            y = y.to(device)

            logits, z = model(x, m)
            loss = criterion(logits, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

            bs = x.size(0)
            total_loss += loss.item() * bs
            total_seen += bs
            total_correct += (logits.argmax(dim=1) == y).sum().item()

            pbar.set_postfix(
                loss=total_loss / max(total_seen, 1),
                acc=total_correct / max(total_seen, 1)
            )

            # one-time sanity check on the first seen batch
            if not did_sanity:
                print("\n=== SANITY CHECK (first batch) ===")
                print("num_classes:", num_classes)
                print("x:", tuple(x.shape), x.dtype)
                print("m:", tuple(m.shape), m.dtype, "mask_sum[0]:", float(m[0].sum().item()))
                print("y:", tuple(y.shape), y.dtype, "min/max:", int(y.min().item()), int(y.max().item()))
                print("logits:", tuple(logits.shape), logits.dtype, "logits[0,:5]:", logits[0,:5].detach().cpu())
                print("z:", tuple(z.shape), z.dtype, "z_norm[0]:", float(z[0].norm().item()))
                assert logits.ndim == 2 and logits.size(1) == num_classes
                assert y.dtype == torch.long
                assert torch.isfinite(z).all()
                print("✅ Sanity checks passed.\n")
                did_sanity = True

        train_loss = total_loss / max(total_seen, 1)
        train_acc  = total_correct / max(total_seen, 1)

        # retrieval validation on categories (same as before)
        proto_classes, proto_mat = compute_class_prototypes(model.encoder, val_support_loader, device=device)
        val_accs = retrieval_eval(model.encoder, val_query_loader, proto_classes, proto_mat, device=device)
        top1 = float(val_accs.get(1, 0.0))

        print(
            f"Epoch {ep:02d} | "
            f"TrainLoss={train_loss:.4f} TrainAcc={train_acc:.4f} | "
            f"ValRetr Top1={val_accs.get(1,0):.4f} Top5={val_accs.get(5,0):.4f} Top10={val_accs.get(10,0):.4f}"
        )

        # early stopping on retrieval Top-1
        if top1 > best_top1 + min_delta:
            best_top1 = top1
            best_epoch = ep
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"Early stopping at epoch {ep} (best Top1={best_top1:.4f} at epoch {best_epoch}).")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"Restored best model from epoch {best_epoch} (Top1={best_top1:.4f}).")

    return model


In [200]:
# =========================
# RUN (Stage-1)
# =========================
device = "cuda" if torch.cuda.is_available() else "mps"
num_classes = df_train["encoded_category"].nunique()

model = train_stage1(
    train_loader=train_loader,
    val_support_loader=val_support_loader,
    val_query_loader=val_query_loader,
    num_classes=num_classes,
    d=256,
    dropout=0.05,
    epochs=30,
    lr=1e-4,
    weight_decay=0.05,
    patience=5,
    min_delta=1e-4,
    device=device
)


Epoch 1/30:   0%|          | 0/477 [00:00<?, ?it/s, acc=0.125, loss=4.05]


=== SANITY CHECK (first batch) ===
num_classes: 58
x: (8, 1024, 51) torch.float32
m: (8, 51) torch.float32 mask_sum[0]: 7.0
y: (8,) torch.int64 min/max: 0 32
logits: (8, 58) torch.float32 logits[0,:5]: tensor([ 0.1577,  0.0536, -0.0300, -0.0284,  0.0473])
z: (8, 256) torch.float32 z_norm[0]: 0.9999999403953552


Epoch 1/30:   3%|▎         | 14/477 [00:00<00:13, 35.05it/s, acc=0.067, loss=4.03] 

✅ Sanity checks passed.



Epoch 01 | TrainLoss=3.9192 TrainAcc=0.1181 | ValRetr Top1=0.1182 Top5=0.2409 Top10=0.3591


...

Epoch 04 | TrainLoss=3.6229 TrainAcc=0.1734 | ValRetr Top1=0.1445 Top5=0.2876 Top10=0.4277


Epoch 05 | TrainLoss=3.5563 TrainAcc=0.1876 | ValRetr Top1=0.1314 Top5=0.2993 Top10=0.4467


Epoch 06 | TrainLoss=3.4969 TrainAcc=0.1942 | ValRetr Top1=0.1431 Top5=0.3066 Top10=0.4496


Epoch 07 | TrainLoss=3.4383 TrainAcc=0.2091 | ValRetr Top1=0.1518 Top5=0.3270 Top10=0.4584


Epoch 08 | TrainLoss=3.3801 TrainAcc=0.2183 | ValRetr Top1=0.1489 Top5=0.3124 Top10=0.4613
Early stopping at epoch 8 (best Top1=0.1577 at epoch 3).
Restored best model from epoch 3 (Top1=0.1577).


In [201]:
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd

# ---------- helpers ----------
def normalize_gloss(g):
    return " ".join(str(g).strip().upper().split())

@torch.no_grad()
def compute_class_prototypes_centered(encoder, loader, device="mps"):
    """
    Build class prototypes on centered embeddings (mean-centered then L2).
    Returns (classes, proto_mat, mu) where mu is the global mean embedding.
    """
    encoder.eval()
    Z_all = []
    Y_all = []

    for x, m, y in loader:
        x = x.to(device); m = m.to(device); y = y.to(device)
        z = encoder(x, m)  # already L2, but we'll re-center anyway
        Z_all.append(z)
        Y_all.append(y)

    Z = torch.cat(Z_all, dim=0)  # (N, d)
    Y = torch.cat(Y_all, dim=0)  # (N,)
    mu = Z.mean(dim=0, keepdim=True)  # (1, d)

    Zc = F.normalize(Z - mu, p=2, dim=-1)

    sums, counts = {}, {}
    for i in range(Zc.size(0)):
        cls = int(Y[i].item())
        if cls not in sums:
            sums[cls] = Zc[i].clone()
            counts[cls] = 1
        else:
            sums[cls] += Zc[i]
            counts[cls] += 1

    classes = sorted(sums.keys())
    proto = []
    for c in classes:
        p = sums[c] / max(counts[c], 1)
        p = F.normalize(p, p=2, dim=-1)
        proto.append(p)

    proto_mat = torch.stack(proto, dim=0)  # (C, d)
    return classes, proto_mat, mu

@torch.no_grad()
def retrieval_eval_centered(encoder, query_loader, proto_classes, proto_mat, mu, device="mps", topk=(1,5,10)):
    """
    Center query embeddings using same mu, then cosine vs proto_mat.
    """
    encoder.eval()
    proto_mat = proto_mat.to(device)
    mu = mu.to(device)

    idx_map = {c:i for i,c in enumerate(proto_classes)}

    hits = {k: 0 for k in topk}
    total = 0

    for x, m, y in query_loader:
        x = x.to(device); m = m.to(device); y = y.to(device)
        z = encoder(x, m)                # (B, d)
        zc = F.normalize(z - mu, p=2, dim=-1)

        sims = zc @ proto_mat.t()        # (B, C)
        total += zc.size(0)

        true_idx = torch.tensor([idx_map[int(t.item())] for t in y], device=device)

        for k in topk:
            pred = sims.topk(k, dim=1).indices
            hits[k] += (pred == true_idx.unsqueeze(1)).any(dim=1).sum().item()

    return {k: hits[k]/max(total,1) for k in topk}

# ---------- build prototype/test FULL (gloss) ----------
# Assumes you already have:
# df_prototype, df_test, df_features, DFVideoDataset, collate_fn, and model (trained)

device = "cuda" if torch.cuda.is_available() else "mps"

df_prototype = df_prototype.copy()
df_test = df_test.copy()

df_prototype["gloss_norm"] = df_prototype["gloss"].apply(normalize_gloss)
df_test["gloss_norm"] = df_test["gloss"].apply(normalize_gloss)

from sklearn.preprocessing import LabelEncoder
gloss_le = LabelEncoder()
df_prototype["encoded_gloss"] = gloss_le.fit_transform(df_prototype["gloss_norm"])

proto_glosses = set(df_prototype["gloss_norm"])
df_test_seen = df_test[df_test["gloss_norm"].isin(proto_glosses)].copy()
df_test_seen["encoded_gloss"] = gloss_le.transform(df_test_seen["gloss_norm"])

print("Dropped test rows (unseen gloss):", df_test.shape[0] - df_test_seen.shape[0])
print("Unique glosses in prototype:", df_prototype["gloss_norm"].nunique())
print("Unique glosses in test_seen:", df_test_seen["gloss_norm"].nunique())

# Build feature tables
proto_rows = []
for uid, enc_g in zip(df_prototype["uid"].values, df_prototype["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    proto_rows.append((uid, tensor, enc_g))
df_proto_full = pd.DataFrame(proto_rows, columns=["uid","I3D_features","encoded_category"])

test_rows = []
for uid, enc_g in zip(df_test_seen["uid"].values, df_test_seen["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    test_rows.append((uid, tensor, enc_g))
df_test_full = pd.DataFrame(test_rows, columns=["uid","I3D_features","encoded_category"])

proto_dataset = DFVideoDataset(df_proto_full, label_col="encoded_category")
test_dataset  = DFVideoDataset(df_test_full,  label_col="encoded_category")

proto_loader = torch.utils.data.DataLoader(proto_dataset, batch_size=64, shuffle=False, collate_fn=collate_fn)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=64, shuffle=False, collate_fn=collate_fn)

# ---------- centered-cosine retrieval ----------
proto_classes, proto_mat, mu = compute_class_prototypes_centered(model.encoder, proto_loader, device=device)
accs = retrieval_eval_centered(model.encoder, test_loader, proto_classes, proto_mat, mu, device=device, topk=(1,5,10))

print("\nTEST (Gloss Retrieval, Centered Cosine):")
print(accs)
print("Top-1 :", round(accs[1], 4))
print("Top-5 :", round(accs[5], 4))
print("Top-10:", round(accs[10], 4))


Dropped test rows (unseen gloss): 0
Unique glosses in prototype: 4765
Unique glosses in test_seen: 1363

TEST (Gloss Retrieval, Centered Cosine):
{1: 0.14398249452954048, 5: 0.16849015317286653, 10: 0.18599562363238512}
Top-1 : 0.144
Top-5 : 0.1685
Top-10: 0.186


In [202]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from tqdm.auto import tqdm

# -------------------------
# 0) Gloss normalization + encoding (prototype + test)
# -------------------------
def normalize_gloss(g):
    return " ".join(str(g).strip().upper().split())

df_prototype = df_prototype.copy()
df_test = df_test.copy()
df_prototype["gloss_norm"] = df_prototype["gloss"].apply(normalize_gloss)
df_test["gloss_norm"] = df_test["gloss"].apply(normalize_gloss)

gloss_le = LabelEncoder()
df_prototype["encoded_gloss"] = gloss_le.fit_transform(df_prototype["gloss_norm"])

proto_glosses = set(df_prototype["gloss_norm"])
df_test_seen = df_test[df_test["gloss_norm"].isin(proto_glosses)].copy()
df_test_seen["encoded_gloss"] = gloss_le.transform(df_test_seen["gloss_norm"])

print("Dropped test rows (unseen gloss):", df_test.shape[0] - df_test_seen.shape[0])
print("Unique glosses in prototype:", df_prototype["gloss_norm"].nunique())
print("Unique glosses in test_seen:", df_test_seen["gloss_norm"].nunique())

# -------------------------
# 1) Build df_proto_full + df_test_full with features
# -------------------------
proto_rows = []
for uid, enc_g in zip(df_prototype["uid"].values, df_prototype["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    proto_rows.append((uid, tensor, enc_g))
df_proto_full = pd.DataFrame(proto_rows, columns=["uid","I3D_features","encoded_gloss"])

test_rows = []
for uid, enc_g in zip(df_test_seen["uid"].values, df_test_seen["encoded_gloss"].values):
    tensor = df_features.loc[df_features["id"] == uid, "I3D_features"].iloc[0].squeeze()
    test_rows.append((uid, tensor, enc_g))
df_test_full = pd.DataFrame(test_rows, columns=["uid","I3D_features","encoded_gloss"])

# -------------------------
# 2) Dataset that returns (x, y) but keeps uid index stable
# -------------------------
class DFVideoDatasetGloss(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        x = np.array(self.df.loc[idx, "I3D_features"])  # (1024, T)
        y = int(self.df.loc[idx, "encoded_gloss"])
        return x, y

# -------------------------
# 3) Make 2-view temporal augmentation (no retraining)
#    - random crop + random time dropout
#    Works on padded tensors (B,1024,Tmax) with mask (B,Tmax)
# -------------------------
def two_view_time_aug(x, m, crop_min=0.6, drop_p=0.1):
    """
    x: (B,1024,T), m: (B,T) float {0,1}
    returns two views (x1,m1,x2,m2)
    """
    B, D, T = x.shape
    device = x.device

    def make_view():
        xv = x.clone()
        mv = m.clone()

        # for each sample, crop within valid length
        for i in range(B):
            valid_idx = torch.where(mv[i] > 0)[0]
            if len(valid_idx) <= 2:
                continue
            L = int(len(valid_idx))
            # crop length
            cL = max(2, int(L * (crop_min + (1.0 - crop_min) * torch.rand(1, device=device).item())))
            if cL >= L:
                continue
            start = int(torch.randint(0, L - cL + 1, (1,), device=device).item())
            keep = valid_idx[start:start+cL]

            # zero out everything except keep
            mask_new = torch.zeros_like(mv[i])
            mask_new[keep] = 1.0
            mv[i] = mv[i] * mask_new
            xv[i] = xv[i] * mv[i].view(1, -1)

        # random dropout on remaining valid frames
        if drop_p > 0:
            drop = (torch.rand(B, T, device=device) < drop_p).float()
            mv = mv * (1.0 - drop)
            xv = xv * mv.view(B, 1, T)

        return xv, mv

    x1, m1 = make_view()
    x2, m2 = make_view()
    return x1, m1, x2, m2

# -------------------------
# 4) Encode a loader into embeddings (optionally 2-view)
# -------------------------
@torch.no_grad()
def encode_loader(model, loader, device, two_view=False):
    model.eval()
    Z1_list, Z2_list = [], []
    Y_list = []

    for x, m, y in tqdm(loader, desc="Encoding", leave=False):
        x = x.to(device)
        m = m.to(device)
        y = y.to(device)

        if not two_view:
            z = model.encoder(x, m)  # (B,d), already normalized
            Z1_list.append(z.detach().cpu())
            Y_list.append(y.detach().cpu())
        else:
            x1, m1, x2, m2 = two_view_time_aug(x, m, crop_min=0.6, drop_p=0.1)
            z1 = model.encoder(x1, m1)
            z2 = model.encoder(x2, m2)
            Z1_list.append(z1.detach().cpu())
            Z2_list.append(z2.detach().cpu())
            Y_list.append(y.detach().cpu())

    Z1 = torch.cat(Z1_list, dim=0)
    Y  = torch.cat(Y_list, dim=0)
    if two_view:
        Z2 = torch.cat(Z2_list, dim=0)
        return Z1, Z2, Y
    return Z1, Y

# -------------------------
# 5) Centered cosine similarity + temperature
# -------------------------
def centered(Z):
    mu = Z.mean(dim=0, keepdim=True)
    Zc = F.normalize(Z - mu, p=2, dim=1)
    return Zc, mu

@torch.no_grad()
def retrieval_uid_self(Zp, Zq, tau=0.1, topk=(1,5,10)):
    """
    Zp: (N,d) prototypes, Zq: (N,d) queries, correct match is same index i.
    """
    N = Zp.shape[0]
    sims = (Zq @ Zp.t()) / tau  # (N,N)
    # make sure we don't trivially match itself if you used same view (we don't)
    true = torch.arange(N, device=sims.device)

    acc = {}
    for k in topk:
        pred = sims.topk(k, dim=1).indices
        hit = (pred == true.unsqueeze(1)).any(dim=1).float().mean().item()
        acc[k] = hit
    return acc

# -------------------------
# 6) Build loaders for prototype + test (gloss labels)
# -------------------------
# IMPORTANT: We want stable ordering for self-retrieval, so shuffle=False
proto_ds = DFVideoDataset(df_proto_full.rename(columns={"encoded_gloss":"encoded_category"}),
                          label_col="encoded_category")
proto_loader = DataLoader(proto_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

test_ds = DFVideoDataset(df_test_full.rename(columns={"encoded_gloss":"encoded_category"}),
                         label_col="encoded_category")
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, collate_fn=collate_fn)

device = "cuda" if torch.cuda.is_available() else "mps"
model = model.to(device)

# -------------------------
# 7) Pseudo-val: 2-view self-retrieval on prototype set → tune tau
# -------------------------
Zp1, Zp2, Yp = encode_loader(model, proto_loader, device=device, two_view=True)

# centered cosine
Zp1c, mu = centered(Zp1)
Zp2c = F.normalize(Zp2 - mu, p=2, dim=1)

taus = [0.03,0.05,0.07,0.1,0.15,0.2,0.3,0.5,1.0]
best_tau, best_top1 = None, -1

print("\nPseudo-val (2-view self-retrieval on prototypes):")
for tau in taus:
    acc = retrieval_uid_self(Zp1c.to(device), Zp2c.to(device), tau=tau, topk=(1,5,10))
    print(f"tau={tau:>4}  acc={acc}")
    if acc[1] > best_top1:
        best_top1 = acc[1]
        best_tau = tau

print("\nBest tau by pseudo-val Top1:", best_tau, "Top1:", best_top1)

# -------------------------
# 8) FINAL: Gloss retrieval on test using full prototype set, centered cosine + best_tau
# -------------------------
# Encode full prototype embeddings (single-view) and test embeddings
Zp, _ = encode_loader(model, proto_loader, device=device, two_view=False)
Zt, Yt = encode_loader(model, test_loader, device=device, two_view=False)

# centered cosine using prototype mean
Zpc, mu = centered(Zp)
Ztc = F.normalize(Zt - mu, p=2, dim=1)

# similarity to all prototypes
sims = (Ztc.to(device) @ Zpc.to(device).t()) / best_tau

# top-k over gloss prototypes (each prototype is one gloss video)
topk = (1,5,10)
accs = {}
true = Yt.to(device)  # encoded_gloss values
proto_labels = df_proto_full["encoded_gloss"].values
proto_labels = torch.tensor(proto_labels, device=device, dtype=torch.long)

# map each test label to matching prototype index (since prototype has exactly 1 per gloss)
# build dict gloss->proto_idx
gloss_to_idx = {}
for i, g in enumerate(proto_labels.tolist()):
    gloss_to_idx[g] = i
true_idx = torch.tensor([gloss_to_idx[int(g.item())] for g in true], device=device)

for k in topk:
    pred_idx = sims.topk(k, dim=1).indices
    hit = (pred_idx == true_idx.unsqueeze(1)).any(dim=1).float().mean().item()
    accs[k] = hit

print("\nTEST (Gloss Retrieval, Centered Cosine + tuned tau):")
print(accs)
print("Top-1 :", round(accs[1], 4))
print("Top-5 :", round(accs[5], 4))
print("Top-10:", round(accs[10], 4))


Dropped test rows (unseen gloss): 0
Unique glosses in prototype: 4765
Unique glosses in test_seen: 1363



Pseudo-val (2-view self-retrieval on prototypes):
tau=0.03  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}
tau=0.05  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}
tau=0.07  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}
tau= 0.1  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}
tau=0.15  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}
tau= 0.2  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}
tau= 0.3  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}
tau= 0.5  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}
tau= 1.0  acc={1: 0.9292759895324707, 5: 0.9603357911109924, 10: 0.9699894785881042}

Best tau by pseudo-val Top1: 0.03 Top1: 0.9292759895324707



TEST (Gloss Retrieval, Centered Cosine + tuned tau):
{1: 0.14398249983787537, 5: 0.16849015653133392, 10: 0.18599562346935272}
Top-1 : 0.144
Top-5 : 0.1685
Top-10: 0.186


In [203]:
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

# -------------------------
# 1) Single-view time augmentation (uses your logic)
# -------------------------
@torch.no_grad()
def one_view_time_aug(x, m, crop_min=0.6, drop_p=0.1):
    """
    x: (B,1024,T), m: (B,T) float {0,1}
    returns one view (xv, mv)
    """
    B, D, T = x.shape
    device = x.device

    xv = x.clone()
    mv = m.clone()

    # crop within valid length
    for i in range(B):
        valid_idx = torch.where(mv[i] > 0)[0]
        if len(valid_idx) <= 2:
            continue
        L = int(len(valid_idx))
        # crop length
        cL = max(2, int(L * (crop_min + (1.0 - crop_min) * torch.rand(1, device=device).item())))
        if cL >= L:
            continue
        start = int(torch.randint(0, L - cL + 1, (1,), device=device).item())
        keep = valid_idx[start:start + cL]

        mask_new = torch.zeros_like(mv[i])
        mask_new[keep] = 1.0
        mv[i] = mv[i] * mask_new
        xv[i] = xv[i] * mv[i].view(1, -1)

    # dropout on remaining valid frames
    if drop_p > 0:
        drop = (torch.rand(B, T, device=device) < drop_p).float()
        mv = mv * (1.0 - drop)
        xv = xv * mv.view(B, 1, T)

    return xv, mv


# -------------------------
# 2) Build prototype matrix from loader (L2 normalized)
# -------------------------
@torch.no_grad()
def build_proto_matrix(encoder, proto_loader, device="mps"):
    encoder.eval()
    sums = {}
    counts = {}

    for x, m, y in tqdm(proto_loader, desc="Building prototypes", leave=False):
        x = x.to(device); m = m.to(device); y = y.to(device)
        z = encoder(x, m)  # (B,d) already L2-normalized in your encoder

        for i in range(z.size(0)):
            cls = int(y[i].item())
            if cls not in sums:
                sums[cls] = z[i].detach().clone()
                counts[cls] = 1
            else:
                sums[cls] += z[i]
                counts[cls] += 1

    classes = sorted(sums.keys())
    proto = []
    for c in classes:
        p = sums[c] / max(counts[c], 1)
        p = F.normalize(p, p=2, dim=-1)
        proto.append(p)

    Zp = torch.stack(proto, dim=0)  # (C,d)
    return classes, Zp


# -------------------------
# 3) Centered cosine helper (optional but you liked it)
# -------------------------
@torch.no_grad()
def centered_cosine_sims(Zq, Zp):
    """
    Zq: (B,d), Zp: (C,d) both L2 normed
    returns sims: (B,C) using mean-centering then re-normalize
    """
    Zq2 = Zq - Zq.mean(dim=1, keepdim=True)
    Zp2 = Zp - Zp.mean(dim=1, keepdim=True)
    Zq2 = F.normalize(Zq2, p=2, dim=-1)
    Zp2 = F.normalize(Zp2, p=2, dim=-1)
    return Zq2 @ Zp2.t()


# -------------------------
# 4) Multi-view query ensemble retrieval
# -------------------------
@torch.no_grad()
def retrieval_eval_multiview(
    encoder,
    test_loader,
    proto_classes,
    Zp,
    device="mps",
    topk=(1,5,10),
    n_views=5,
    crop_min=0.6,
    drop_p=0.1,
    agg="max",            # "max" or "mean"
    use_centered=True
):
    """
    For each test batch:
      - sample n_views augmented views
      - embed each view
      - compute sims to prototypes
      - aggregate sims across views -> final sims
      - compute Top-K
    """
    encoder.eval()
    Zp = Zp.to(device)  # (C,d)

    idx_map = {c:i for i,c in enumerate(proto_classes)}

    hits = {k:0 for k in topk}
    total = 0

    for x, m, y in tqdm(test_loader, desc="Test multi-view retrieval", leave=False):
        x = x.to(device); m = m.to(device); y = y.to(device)
        B = x.size(0)

        # convert true labels -> prototype indices (skip samples whose label not in proto_classes)
        keep_mask = torch.tensor([int(t.item()) in idx_map for t in y], device=device, dtype=torch.bool)
        if keep_mask.sum().item() == 0:
            continue

        x = x[keep_mask]; m = m[keep_mask]; y = y[keep_mask]
        B = x.size(0)
        true_idx = torch.tensor([idx_map[int(t.item())] for t in y], device=device)

        # collect sims over views: (n_views, B, C)
        sims_views = []

        for _ in range(n_views):
            xv, mv = one_view_time_aug(x, m, crop_min=crop_min, drop_p=drop_p)
            Zq = encoder(xv, mv)  # (B,d)

            if use_centered:
                sims = centered_cosine_sims(Zq, Zp)  # (B,C)
            else:
                sims = Zq @ Zp.t()

            sims_views.append(sims)

        sims_views = torch.stack(sims_views, dim=0)  # (V,B,C)

        if agg == "max":
            sims_final = sims_views.max(dim=0).values  # (B,C)
        elif agg == "mean":
            sims_final = sims_views.mean(dim=0)        # (B,C)
        else:
            raise ValueError("agg must be 'max' or 'mean'")

        total += B

        for k in topk:
            pred = sims_final.topk(k, dim=1).indices  # (B,k)
            hits[k] += (pred == true_idx.unsqueeze(1)).any(dim=1).sum().item()

    return {k: hits[k] / max(total,1) for k in topk}, total


# -------------------------
# 5) Example usage (paste & run)
# -------------------------
device = "cuda" if torch.cuda.is_available() else "mps"

# Build prototypes ONCE from full prototype set (gloss or category, depending on your proto_loader labels)
proto_classes, Zp = build_proto_matrix(model.encoder, proto_loader, device=device)

# Baseline single-view (n_views=1)
acc1, kept1 = retrieval_eval_multiview(
    model.encoder, test_loader,
    proto_classes, Zp,
    device=device,
    n_views=1,
    crop_min=0.6,
    drop_p=0.1,
    agg="max",
    use_centered=True
)
print("Single-view centered:", acc1, "Kept=", kept1)

# Multi-view ensemble (try these first)
for V in [3, 5, 10]:
    acc, kept = retrieval_eval_multiview(
        model.encoder, test_loader,
        proto_classes, Zp,
        device=device,
        n_views=V,
        crop_min=0.6,   # try 0.5 or 0.7 also
        drop_p=0.1,     # try 0.05 / 0.15
        agg="max",      # max usually best
        use_centered=True
    )
    print(f"{V}-view centered max:", acc, "Kept=", kept)


Single-view centered: {1: 0.13435448577680525, 5: 0.16280087527352297, 10: 0.17680525164113786} Kept= 2285


3-view centered max: {1: 0.1409190371991247, 5: 0.16323851203501094, 10: 0.1785557986870897} Kept= 2285


5-view centered max: {1: 0.14179431072210066, 5: 0.16805251641137856, 10: 0.18424507658643327} Kept= 2285


10-view centered max: {1: 0.1435448577680525, 5: 0.16805251641137856, 10: 0.18336980306345732} Kept= 2285
